Mounting Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CC_NEWS dataset - Do not run from here again.

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset, load_from_disk
import os

In [ ]:
import os
save_path = "/content/drive/MyDrive/gdcm/data/cc_news"
os.makedirs(save_path, exist_ok=True)

In [ ]:
dataset = load_dataset("vblagoje/cc_news")

In [ ]:
dataset.save_to_disk(save_path)

In [ ]:
# see first five rows of data

from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/gdcm/data/cc_news")

df = dataset["train"].to_pandas()
# print(df)

In [ ]:
df.info()

In [ ]:
#took couple of seconds to load

In [ ]:
import pandas as pd

In [ ]:
# Looking at basic info of CC_News dataset

In [ ]:
print("Shape:", df.shape)
df.info()
df.isnull().sum()
(df.isnull().sum() / len(df)) * 100
df

In [ ]:
print("Column names:", df.columns.tolist())

In [ ]:
# Need to look  into dataset once to see potential issues, null, missing, text length etc.

In [ ]:
print("\n Empty string values:")
for col in df.columns:
    empty_count = (df[col].astype(str).str.strip() == "").sum()
    print(f"{col}: {empty_count}")

In [ ]:
# Text length analysis
df["text_length"] = df["text"].astype(str).str.len()
# print(df["text_length"])
print(df["text_length"].describe())

In [ ]:
print("\n Very short texts (<50 chars):", (df["text_length"] < 50).sum())
print("Very long texts (>10000 chars):", (df["text_length"] > 10000).sum())

In [ ]:
# duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())
# Duplicate based on key columns (stronger check)
print("Duplicate text:", df["text"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

# URL validity
print("\n URL format check:")
print(df["url"].astype(str).str.startswith("http").value_counts())

# Image URL validity
print("\n Image URL format check:")
print(df["image_url"].astype(str).str.startswith("http").value_counts())

# Domain distribution
print("\n Top domains:")
print(df["domain"].value_counts().head(10))


# 8. Date consistency


# Convert the "date" column to datetime format
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("\nDate issues:")
print("Invalid dates:", df["date"].isnull().sum())
print("Date range:", df["date"].min(), "to", df["date"].max())


check for empty strings

In [ ]:
(df["text"].str.strip() == "").sum()

In [ ]:
for col in df.columns:
    empty_count = (df[col].astype(str).str.strip() == "").sum()
    print(f"{col}: {empty_count} empty values")

I think from a text analysis perspective we need to drop some columns for pre-processing and also fix some issues.

In [ ]:
# Empty titles
df[df["title"].astype(str).str.strip() == ""][["title", "text"]].head()

In [ ]:
# missing descriptions in the column

df[df["description"].isnull()][["title", "text", "description"]].head(5)

In [ ]:
df[df["text"].duplicated()][["title","text", "domain","description"]].head(10)

In [ ]:
cols = ["title", "text", "domain", "description"]

df_sample = df[cols].sample(25, random_state=42)
df_sample

In [ ]:
# looking into exact and partial matches
# 1. Exact duplicates (all columns match)
exact_df_dup = df.duplicated().sum()

# 2. Text-only duplicates (same story, different metadata/URL)
text_dup = df.duplicated(subset=['text']).sum()

# 3. Title-only duplicates (potentially different versions of the same story)
title_dup = df.duplicated(subset=['title']).sum()

print(f"Exact row duplicates: {exact_df_dup}")
print(f"Duplicate story texts: {text_dup}")
print(f"Duplicate titles: {title_dup}")

checking language distbribution

In [ ]:
!pip install langid

import pandas as pd
import langid
from collections import Counter

# 1. To save time (800k is a lot), we only check UNIQUE texts
# This ensures we don't detect the same language 141,437 extra times
unique_texts = df['text'].drop_duplicates().dropna()

print(f"Analyzing language for {len(unique_texts):,} unique stories...")

def quick_lang(t):
    # Standardize and shorten for speed - first 500 chars is enough for detection
    sample = str(t)[:500].replace("\n", " ")
    lang, _ = langid.classify(sample)
    return lang

# 2. Apply detection
# This will take a few minutes. You can test on .sample(1000) first if you're in a hurry.
text_languages = unique_texts.apply(quick_lang)

# 3. Calculate Distribution
lang_dist = text_languages.value_counts(normalize=True).head(10) * 100

# print("\n--- Language Distribution (Unique Texts) ---")
# for lang, percent in lang_dist.items():
#     print(f"{lang.upper()}: {percent:.2f}%")

# # 4. Check for "Title vs Text" mismatches on a small sample
# # (Optional: helpful to see if titles are English but body is not)
# print("\nQuick check: Is the 'title' language usually the same?")
# sample_check = df.sample(1000).copy()
# sample_check['t_lang'] = sample_check['title'].apply(quick_lang)
# sample_check['b_lang'] = sample_check['text'].apply(quick_lang)
# mismatch_pct = (sample_check['t_lang'] != sample_check['b_lang']).mean() * 100
# print(f"Title/Body language mismatch rate in sample: {mismatch_pct:.2f}%")

So this shows that we have other languages in the dataset, we need only English as both GDCM, CTCBM methods are using English only datasets.
Also, we need to employ more data cleaning methods before we can actually prepare the config file and data file for saving a clean, pre-processed file in our drive.

**Data Cleaning for whole dataset**

In [ ]:
# Select the first 100 rows and the specific columns of interest
top_100_inspect = df[['title', 'text', 'description']].head(100)

# We use .style to left-align the text and add some padding for readability
# This is much better than a standard print() for long news articles
top_100_inspect.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'vertical-align': 'top'
})

In [ ]:
import pandas as pd

# 1. Get the absolute Maximum Lengths
max_text = df['text'].str.len().max()
max_desc = df['description'].str.len().max()
print(f"--- Maximum Lengths ---")
print(f"Max Text:        {max_text:,} characters")
print(f"Max Description: {max_desc:,} characters")

In [ ]:
min_text = df['text'].str.len().min()
min_desc = df['description'].str.len().min()

print(f"--- Minimum Lengths ---")
print(f"Min Text:        {min_text:,} characters")
print(f"Min Description: {min_desc:,} characters")

checking the rows which have long text chain more than 100k in character

In [ ]:
# 1. Sort by length to find the absolute longest 'text' rows
longest_text_rows = df.assign(text_len=df['text'].str.len()).sort_values('text_len', ascending=False).head(5)

# 2. Sort by length to find the absolute longest 'description' rows
longest_desc_rows = df.assign(desc_len=df['description'].str.len()).sort_values('desc_len', ascending=False).head(5)

print("--- INSPECTING TOP 5 LONGEST 'TEXT' ROWS ---")
for i, (idx, row) in enumerate(longest_text_rows.iterrows()):
    print(f"\n[Text Rank {i+1}] Length: {len(str(row['text'])):,} | Domain: {row['domain']}")
    print(f"Snippet (First 500 chars): {str(row['text'])[:500]}...")
    print("-" * 30)

print("\n" + "="*50 + "\n")

print("--- INSPECTING TOP 5 LONGEST 'DESCRIPTION' ROWS ---")
for i, (idx, row) in enumerate(longest_desc_rows.iterrows()):
    print(f"\n[Desc Rank {i+1}] Length: {len(str(row['description'])):,} | Domain: {row['domain']}")
    print(f"Snippet (First 500 chars): {str(row['description'])[:500]}...")
    print("-" * 30)

In [ ]:
# 1. Get the absolute Maximum Lengths
max_title = df['title'].str.len().max()
# max_desc = df['description'].str.len().max()
print(f"--- Maximum Lengths ---")
print(f"Max Title:        {max_title:,} characters")
# print(f"Max Description: {max_desc:,} characters")

min_title = df['title'].str.len().min()

print(f"--- Minimum Lengths ---")
print(f"Min Title:        {min_title:,} characters")

Further Data cleaning : Removing duplicate text, null, empty rows, rows which have other languages, duplicates rows, also need to check which column will serve as a guide to our labels mapping and concepts , then we need to look into pre-processing of that cokumn that may yield labels, and concepts this has to be done after normalization applied on the dataset so the sequence looks like this, **empty>>null>> non-english>> dedup>> lowercasing>> html removal>> normalization>> length check for text>> then comes label mapping >> processingcolumn for label mapping, then we move to tokenization,** and train, test val split etc after processing of label columns we will stop and add an explanantory paragrap based on our observation.

So rules are needed for each column separately, because each column is raw right now, and each column we are considering has different issue that needs to be looked into so only after exhaustive data cleaning and rules application and everything we can start with config files. so each column needs thorough cleaning.

So I think I will keep title and text and leave description, we can study text, truncate it, remove some words in it, keep it free flowing and truncate the character from front and back, we need to remove symbols, remvoe html, and remove a lot of things like

For actual data cleaning, first we make a copy of data, drop irrelevant columns because I dont htink date, url, image url will be used, I will be using tittle, text, description, Im debating whether to keep text or description , im leaning toeards keeping text column as it has more context.



# **Exploratory analysis of CC_NEWS**

In [ ]:
import pandas as pd
import re

# 1. Create a deep copy and drop columns like url, image_url, date
df_cc_news = df.copy()
df_cc_news.drop(columns=['url', 'image_url', 'date'], errors='ignore', inplace=True)


# Symbol Walls: 4 or more repeating symbols like ----, ::::, ____
symbol_wall_pattern = r'([!@#$%^&*|\\:;?._=-])\1{3,}'
# Non-ASCII: Anything outside standard English characters
non_ascii_pattern = r'[^\x00-\x7F]'

def inspect_junk(data, column):
    print(f"\n{'='*30}\nINSPECTING COLUMN: {column}\n{'='*30}")

    # Identify rows with walls
    has_wall = data[column].str.contains(symbol_wall_pattern, na=False, regex=True)
    wall_rows = data[has_wall]

    print(f"Total rows with Symbol Walls: {len(wall_rows):,}")
    if not wall_rows.empty:
        print(f"\nTop 5 Examples of 'Symbol Walls' in {column}:")
        for i, val in enumerate(wall_rows[column].head(5)):
            # Show snippet of the wall
            match = re.search(symbol_wall_pattern, str(val))
            snippet = str(val)[:300] + "..." if len(str(val)) > 300 else str(val)
            print(f"[{i+1}] Pattern found: '{match.group(0)}' | Full context snippet:\n{snippet}\n")

    # Identify rows with non-ASCII
    has_non_ascii = data[column].str.contains(non_ascii_pattern, na=False, regex=True)
    non_ascii_rows = data[has_non_ascii]

    print(f"\nTotal rows with Non-ASCII characters: {len(non_ascii_rows):,}")
    if not non_ascii_rows.empty:
        print(f"\nTop 5 Examples of 'Non-ASCII' in {column}:")
        for i, val in enumerate(non_ascii_rows[column].head(5)):
            # Find which symbols specifically
            junk_found = set(re.findall(non_ascii_pattern, str(val)))
            snippet = str(val)[:300] + "..." if len(str(val)) > 300 else str(val)
            print(f"[{i+1}] Symbols found: {junk_found} | Snippet:\n{snippet}\n")

# 3. Run inspection for all columns
for col in ['text', 'title', 'description']:
    if col in df_cc_news.columns:
        inspect_junk(df_cc_news, col)

In [ ]:
from collections import Counter
import re

# 1. Ensure we are using your specific copy
df_clean = df_cc_news.copy()

def fish_out_symbols(data, column, sample_size=50000):
    # Sample to keep it fast on 800k rows
    sample_text = " ".join(data[column].fillna('').sample(sample_size).astype(str))

    # REGEX: Catch everything that is NOT a letter, number, or basic punctuation
    # This will catch math symbols, currency, non-English scripts, and weird artifacts
    weird_stuff = re.findall(r'[^a-zA-Z0-9\s\.,!\?\(\)\-\"\']', sample_text)

    symbol_counts = Counter(weird_stuff)

    print(f"\n--- TOP 30 'NON-STANDARD' SYMBOLS IN {column} ---")
    print(f"{'Symbol':<10} | {'Count':<10} | {'Unicode Hex':<10}")
    print("-" * 40)
    for sym, count in symbol_counts.most_common(30):
        print(f"{sym:<10} | {count:<10} | {hex(ord(sym)):<10}")

# Run the fishing net
fish_out_symbols(df_clean, 'text')

In [ ]:
def inspect_edge_walls(data, column):
    # Regex for 3+ repeating symbols at the very start (^) or very end ($)
    start_wall_pattern = r'^[!@#$%^&*|\\:;?._=-]{3,}'
    end_wall_pattern = r'[!@#$%^&*|\\:;?._=-]{3,}$'

    print(f"\n{'='*50}")
    print(f"EDGE INSPECTION: {column}")
    print(f"{'='*50}")

    # --- STARTING WALLS ---
    starts_with_wall = data[data[column].str.contains(start_wall_pattern, na=False, regex=True)]
    print(f"Total rows starting with a Wall: {len(starts_with_wall):,}")

    if not starts_with_wall.empty:
        print("\nTOP 5 EXAMPLES (START OF ROW):")
        for i, val in enumerate(starts_with_wall[column].head(5)):
            # Show the first 200 chars to see the 'Wall' in context
            print(f"[{i+1}] {str(val)[:200]}...")

    # --- ENDING WALLS ---
    ends_with_wall = data[data[column].str.contains(end_wall_pattern, na=False, regex=True)]
    print(f"\nTotal rows ending with a Wall: {len(ends_with_wall):,}")

    if not ends_with_wall.empty:
        print("\nTOP 5 EXAMPLES (END OF ROW):")
        for i, val in enumerate(ends_with_wall[column].head(5)):
            # Show the last 200 chars
            print(f"[{i+1}] ...{str(val)[-200:]}")

# Run for both major text columns
inspect_edge_walls(df_clean, 'text')
inspect_edge_walls(df_clean, 'description')

to see the starting rows of the data  where it starts from -------

In [ ]:
import pandas as pd
import re

# 1. Define the pattern for 3+ repeating symbols at the VERY START
start_wall_pattern = r'^[!@#$%^&*|\\:;?._=-]{3,}'

# 2. Filter df_cc_news for these specific rows
starts_with_wall = df_cc_news[df_cc_news['text'].str.contains(start_wall_pattern, na=False, regex=True)].copy()

print(f"Total rows starting with a Wall: {len(starts_with_wall):,}")

# 3. Display the top 10 rows for deep inspection
if not starts_with_wall.empty:
    print("\n--- DEEP INSPECTION: TOP 10 ROWS STARTING WITH SYMBOLS ---")
    # We display Title and the beginning of Text to check for 'Signal'
    for i, (idx, row) in enumerate(starts_with_wall[['title', 'text']].head(10).iterrows()):
        print(f"Row Index: {idx}")
        print(f"Title: {row['title']}")
        # Show the first 250 chars to see the wall and what follows it
        print(f"Text Snippet: {str(row['text'])[:250]}...")
        print("-" * 50)
else:
    print("No rows found starting with that specific symbol pattern.")

when we look at

In [ ]:
import pandas as pd
import re

# 1. Pattern for 3+ repeating symbols at the very start
start_wall_pattern = r'^[!@#$%^&*|\\:;?._=-]{3,}'

# 2. Filter df_cc_news for these specific rows
starts_with_wall = df_cc_news[df_cc_news['text'].str.contains(start_wall_pattern, na=False, regex=True)].copy()

# 3. Create a display-friendly version of the Top 10
# We use .str.slice to see the exact 'Wall' and the content following it
inspection_table = starts_with_wall[['title', 'text']].head(10).copy()
inspection_table['text_snippet'] = inspection_table['text'].str.slice(0, 250) + "..."

# 4. Display in a clean Markdown table format
from IPython.display import display, HTML

print(f"Total rows starting with a Symbol Wall: {len(starts_with_wall):,}")
display(inspection_table[['title', 'text_snippet']])

In [ ]:
import pandas as pd

# 1. Set display options so the text isn't cut off in the table
pd.set_option('display.max_colwidth', 200)

# 2. List of indices you identified
target_indices = [337, 928, 1101, 1859, 2782, 2847, 2910, 7488, 7578, 7916]

# 3. Create the table view from your 'df_cc_news' copy
# We select the index, title, and a snippet of the text
table_view = df_cc_news.loc[target_indices, ['title', 'text']].copy()

# 4. Display the result in a markdown table format
print(table_view.to_markdown())

In [ ]:
import pandas as pd

# 1. Set to 'None' so you see the FULL text within the horizontal table
pd.set_option('display.max_colwidth', None)

# 2. Call the row in the normal horizontal view
# (Using double brackets [[337]] ensures it stays as a table/DataFrame view)
df_cc_news.loc[[337], ['title', 'text', 'description']]

counting how many such rows are there

In [ ]:
import pandas as pd
import re

# 1. Broad pattern for any 3+ repeating symbols at the very start (^)
# This catches: ---, ___, \\\, |||, ..., !!!, etc.
start_wall_pattern = r'^[!@#$%^&*|\\:;?._=-]{3,}'

# 2. Identify the rows in your df_cc_news copy
has_wall = df_cc_news['text'].str.contains(start_wall_pattern, na=False, regex=True)

# 3. Calculate and print the totals
total_problem_rows = has_wall.sum()
percentage = (total_problem_rows / len(df_cc_news)) * 100

print(f"--- Dataset Audit: Symbol Wall Headers ---")
print(f"Total rows starting with a Symbol Wall: {total_problem_rows:,}")
print(f"Percentage of dataset affected:         {percentage:.2f}%")

LOOKING AT ENDING WALL pattersn

In [ ]:
import pandas as pd

# 1. Broad pattern for 3+ repeating symbols at the VERY END ($) of the text
end_wall_pattern = r'[!@#$%^&*|\\:;?._=-]{3,}$'

# 2. Filter the dataframe
# We use .copy() to ensure we can manipulate this subset without warnings
ends_with_wall = df_cc_news[df_cc_news['text'].str.contains(end_wall_pattern, na=False, regex=True)].copy()

# 3. Print the total count
print(f"Total rows ending with a Wall: {len(ends_with_wall):,}")

# 4. Display the Top 10 rows in the normal DataFrame view
# I am selecting just title and text for clarity
ends_with_wall[['title', 'text']].head(10)

In [ ]:
import pandas as pd

# 1. Set display to show the full text content
pd.set_option('display.max_colwidth', None)

# 2. Define your specific indices
target_indices = [287, 291, 786, 1878, 2217]

# 3. Call the rows in the standard horizontal view
df_cc_news.loc[target_indices, ['title', 'text']]

In [ ]:
import pandas as pd

# 1. Identify the 10,558 rows ending with a Wall
end_wall_pattern = r'[!@#$%^&*|\\:;?._=-]{3,}$'
mask_end_wall = df_cc_news['text'].str.contains(end_wall_pattern, na=False, regex=True)

# 2. Within that subset, how many have the '.\n' artifact anywhere in the text?
has_newline_artifact = df_cc_news['text'].str.contains(r'\.\n', na=False, regex=True)

# 3. Calculate the intersection
target_rows = df_cc_news[mask_end_wall & has_newline_artifact]

print(f"Total rows ending with a Wall: {mask_end_wall.sum():,}")
print(f"Of those, rows containing the '.\n' artifact: {len(target_rows):,}")

# 4. View a sample to see if it's all "Tutorial" style like the Excel example
target_rows[['title', 'text']].head(15)

Looking at title column unique vlaues , any patterns that may exsit and geenral info that we can get an idea of


In [ ]:
# See the top 20 most frequent titles
top_titles = df_cc_news['title'].value_counts().head(20)

print("--- Top Recurring Titles ---")
print(top_titles)

In [ ]:
total_rows = len(df_cc_news)
unique_titles = df_cc_news['title'].nunique()
dupe_count = total_rows - unique_titles

print(f"Total Rows: {total_rows:,}")
print(f"Unique Titles: {unique_titles:,}")
print(f"Duplicate Titles: {dupe_count:,} ({(dupe_count/total_rows)*100:.2f}%)")

In [ ]:
# Replace 'Business Briefs' with whatever title showed up most in Step 1
example_title = "Business Highlights"
samples = df_cc_news[df_cc_news['title'] == example_title]['text'].head(5)

print(f"--- Samples for Title: {example_title} ---")
for i, t in enumerate(samples):
    print(f"Sample {i+1}: {t[:200]}...\n")

In [ ]:
# 1. Exact Content Duplicates (Title AND Text are identical)
exact_matches = df_cc_news.duplicated(subset=['title', 'text']).sum()

# 2. Template Duplicates (Same Title, but DIFFERENT Text)
# We find all title dupes and subtract the exact matches
template_dupes = 156534 - exact_matches

print(f"--- Duplicate Breakdown ---")
print(f"Exact Story Matches: {exact_matches:,} (KILL these)")
print(f"Template/Recurring Titles: {template_dupes:,} (INSPECT these)")

Inspecting the 'Exact matches'

In [ ]:
# 1. Identify the rows that have EXACT duplicates (Title + Text)
# This creates a dataframe of just the 'Repeated' stories
dupe_groups = df_cc_news[df_cc_news.duplicated(subset=['title', 'text'], keep=False)]

# 2. See which stories are repeated the MOST
# This helps identify if it's a specific 'Type' of news (e.g., Stock Alerts)
print("--- Top 50 Most Repeated Exact Stories ---")
print(dupe_groups['title'].value_counts().head(50))

# 3. Look at a specific example of a highly repeated story
top_repeated_title = dupe_groups['title'].value_counts().index[0]
example_text = dupe_groups[dupe_groups['title'] == top_repeated_title]['text'].iloc[0]

# print(f"\n--- Content of Most Repeated Story: '{top_repeated_title}' ---")
# print(example_text[:500] + "...")

Looking to see exact matchs just verifying the titles and text

In [ ]:
from IPython.display import display

# 1. Find the exact matches (Title + Text)
exact_matches = df_cc_news.groupby(['title', 'text']).size().reset_index(name='count')

# 2. Sort by count and take the top 50 (to give you a good look)
top_offenders_df = exact_matches.sort_values(by='count', ascending=False).head(50)

# 3. Use the native notebook display for the "normal" view
print("Top 50 Structural Noise / Duplicate Candidates:")
display(top_offenders_df)

In [ ]:
# 1. Define the specific title we want to inspect
target_title = "Dementia support"

# 2. Filter the dataframe for this title and show the first 5 rows
# We use .head(5) to get the sample
inspected_rows = df_cc_news[df_cc_news['title'] == target_title].head(5)

# 3. Display in the normal dataframe view
from IPython.display import display
display(inspected_rows)

In [ ]:
# 1. Define the specific title we want to inspect
target_title = "Get Inspired Magazine"

# 2. Filter the dataframe for this title and show the first 5 rows
# We use .head(5) to get the sample
inspected_rows = df_cc_news[df_cc_news['title'] == target_title].head(5)

# 3. Display in the normal dataframe view
from IPython.display import display
display(inspected_rows)

In [ ]:
# This tells you how many unique 'Stories' are being repeated
num_unique_dupe_stories = dupe_groups.groupby(['title', 'text']).ngroups

print(f"Total rows in dupe_groups: {len(dupe_groups):,}")
print(f"Number of distinct duplicate groups: {num_unique_dupe_stories:,}")

# Example: If the San Bernardino story is 1 group, and 'Subscribe to Read' is 1 group...
# This number tells you exactly how many 'Rows' would remain if you did a 'Keep First' drop.

## **Data Cleaning: CC_NEWS**

## Filter and remove

first looking at the empty rows, text and title and both, then we delete it

In [ ]:
# 1. Check for True Nulls (NaN)
nan_text = df_cc_news['text'].isna().sum()
nan_title = df_cc_news['title'].isna().sum()
nan_both = (df_cc_news['text'].isna() & df_cc_news['title'].isna()).sum()

print(f"--- NULL (NaN) REPORT ---")
print(f"Only Text is NaN:  {nan_text - nan_both:,}")
print(f"Only Title is NaN: {nan_title - nan_both:,}")
print(f"Both are NaN:      {nan_both:,}")


In [ ]:
# 2. Check for Empty Strings (including whitespace)
# We use .fillna('') first so .str.strip() doesn't crash on NaNs
empty_text = (df_cc_news['text'].fillna('').str.strip() == "").sum()
empty_title = (df_cc_news['title'].fillna('').str.strip() == "").sum()
empty_both = ((df_cc_news['text'].fillna('').str.strip() == "") &
              (df_cc_news['title'].fillna('').str.strip() == "")).sum()
print(f"\n--- EMPTY STRING REPORT ---")
print(f"Only Text is Empty:  {empty_text - empty_both:,}")
print(f"Only Title is Empty: {empty_title - empty_both:,}")
print(f"Both are Empty:      {empty_both:,}")


print(f"\n--- TOTAL ROWS TO BE PURGED ---")
total_junk = (df_cc_news['text'].isna() | df_cc_news['title'].isna() |
              (df_cc_news['text'].fillna('').str.strip() == "") |
              (df_cc_news['title'].fillna('').str.strip() == "")).sum()
print(f"Grand Total: {total_junk:,}")

inspecting before we delete it

In [ ]:
# 1. Create the filter for "Empty Title" rows
# This catches NaN, empty strings '', and rows with just spaces '   '
empty_title_mask = df_cc_news['title'].isna() | (df_cc_news['title'].str.strip() == "")

# 2. Select the first 5 rows and only the columns we care about
# We use .copy() to avoid SettingWithCopy warnings
inspect_df = df_cc_news[empty_title_mask].head(5)[['title', 'text']].copy()

# 3. Clean up the display:
# Since titles are empty, we'll fill them with a visible label for the table
inspect_df['title'] = inspect_df['title'].fillna('[NAN]').replace('', '[EMPTY STRING]')

# 4. Show the table
from IPython.display import display
display(inspect_df)

In [ ]:
import pandas as pd
from IPython.display import display

# 1. Lift the "Truncation" limit for this cell
pd.set_option('display.max_colwidth', None)

# 2. Filter for the 331 "Empty Title" rows
empty_title_mask = df_cc_news['title'].isna() | (df_cc_news['title'].str.strip() == "")
full_inspect_df = df_cc_news[empty_title_mask].head(5)[['title', 'text']]

# 3. Show the table (Now with 100% of the text visible)
display(full_inspect_df)

# 4. Reset the option so your other tables don't become massive
pd.reset_option('display.max_colwidth')

In [ ]:
# 1. Identify the indices of the rows with empty/NaN titles
# This targets ONLY the 331 rows we found earlier
rows_to_drop = df_cc_news[
    (df_cc_news['title'].isna()) |
    (df_cc_news['title'].str.strip() == "")
].index

# 2. Delete those specific rows from the original DataFrame
# 'inplace=True' modifies the dataframe directly without needing to reassign it
df_cc_news.drop(index=rows_to_drop, inplace=True)

# 3. Confirmation
print(f"Purged {len(rows_to_drop)} rows with empty titles.")
print(f"New total: {len(df_cc_news):,}")


In [ ]:
## renumber the index again

df_cc_news.reset_index(drop=True, inplace=True)

Removing those rows where data is non-English

In [ ]:
!pip install langdetect

In [ ]:
import pandas as pd
import langid
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# -------------------------------
# Functions (OUTSIDE main block)
# -------------------------------

def quick_lang(t):
    sample = str(t)[:100].replace("\n", " ")
    lang, _ = langid.classify(sample)
    return lang

def parallel_lang_detect(texts, num_workers=None):
    if num_workers is None:
        num_workers = max(cpu_count() - 1, 1)

    texts_list = texts.tolist()

    with Pool(num_workers) as pool:
        results = list(tqdm(pool.imap(quick_lang, texts_list), total=len(texts_list)))

    return pd.Series(results, index=texts.index)

# -------------------------------
# MAIN EXECUTION BLOCK
# -------------------------------
if __name__ == "__main__":

    # Step 1: Deduplicate
    unique_texts = df_cc_news['text'].drop_duplicates().dropna()
    print(f"Unique texts: {len(unique_texts):,}")

    # Step 2: Parallel detection
    text_languages = parallel_lang_detect(unique_texts)

    # Step 3: Map back
    lang_map = dict(zip(unique_texts, text_languages))
    df_cc_news['lang'] = df_cc_news['text'].map(lang_map)

    # Step 4: Filter
    non_en_mask = df_cc_news['lang'] != 'en'

    print(f"\nNon-English rows: {non_en_mask.sum()}")
    print(df_cc_news.loc[non_en_mask, ['lang', 'text']].head(25))

    # # Step 5: Clean dataset
    # df_cc_news_clean = df_cc_news.loc[~non_en_mask].copy()

    # print(f"\nFinal rows: {len(df_cc_news_clean)}")

In [ ]:
  print(df_cc_news.loc[non_en_mask, ['lang', 'text']].head(100))

In [ ]:
indices = [6326, 6327, 6328, 6332, 6335]

for i in indices:
    print(f"\n--- Row {i} ---")
    print("Lang:", df_cc_news.loc[i, 'lang'])
    print("Title:", df_cc_news.loc[i, 'title'])
    print("Text:\n", df_cc_news.loc[i, 'text'])

  We saw that we cannot simply delete the text in non english language as the starting letters maybe in some other language but then the rest of it could be in english, so we need to loook into this and come up with a rule, we also need to study such cases thoroughly. Text cleaning and removal of phrases, letter, number sis required.

Looking for duplicate rows.

In [ ]:
dup_count = df_cc_news.duplicated(subset=['title', 'text']).sum()
print("Number of duplicate rows:", dup_count)

In [ ]:
duplicates = df_cc_news[
    df_cc_news.duplicated(subset=['title', 'text'], keep=False)
]

duplicates[['title', 'text']].head(10)

In [ ]:
# 1. Filter for all rows that have at least one duplicate
duplicates = df_cc_news[df_cc_news.duplicated(subset=['title', 'text'], keep=False)]

# 2. Sort by the columns you care about to group them
duplicates_sorted = duplicates.sort_values(by=['title', 'text'])

# 3. View the top results
duplicates_sorted[['title', 'text']].head(25)

In [ ]:
# 1. Define what "Missing Title" means (NaN or empty string)
missing_title_mask = df_cc_news['title'].isna() | (df_cc_news['title'] == "")

# 2. Identify ALL instances of duplicated text within that "No Title" group
# keep=False marks every single row in a duplicate pair/group as True
all_dupe_no_title_mask = df_cc_news[missing_title_mask].duplicated(subset=['text'], keep=False)

# 3. Get the indices for these rows
indices_to_wipe = df_cc_news[missing_title_mask][all_dupe_no_title_mask].index

print(f"Number of rows to be completely deleted: {len(indices_to_wipe)}")

We will drop the rows where there is No Title but ducplicated texts

In [ ]:
# Drop all identified indices
df_cc_news_purged = df_cc_news.drop(index=indices_to_wipe)

print(f"Original Row Count: {len(df_cc_news)}")
print(f"New Row Count: {len(df_cc_news_purged)}")

we look at the reverse scenario, no text but title present and duplicated

In [ ]:
# 1. Mask for: Title is NOT missing AND Text IS missing
has_title_no_text_mask = df_cc_news['title'].notna() & (df_cc_news['title'] != "") & \
                         (df_cc_news['text'].isna() | (df_cc_news['text'] == ""))

# 2. Identify ALL instances of these duplicates (keep=False)
dupe_no_text_all = df_cc_news[has_title_no_text_mask].duplicated(subset=['title'], keep=False)

# 3. Get the count
count_to_purge = dupe_no_text_all.sum()

print(f"Rows with a title but MISSING/EMPTY text: {has_title_no_text_mask.sum()}")
print(f"Of those, how many are duplicates of each other: {count_to_purge}")

## Remove (Structural Issues)

We remove duplicated data -purging rows which are noise of no value,

We delete the stories  which are duplicates and  appear more than 40 times, we will only keep the first occurence and delete the rest in the code below

In [ ]:
# 1. Identify stories repeating more than 40 times
# We group by Title + Text to see how often each unique story appears
story_counts = df_cc_news_purged.groupby(['title', 'text']).size().reset_index(name='count')
high_freq_trash = story_counts[story_counts['count'] > 40]

# 2. Match those 'Trash' keys back to the main dataframe
# This finds every physical row that belongs to one of these >40 groups
trash_keys = high_freq_trash[['title', 'text']]
all_instances = df_cc_news_purged.reset_index().merge(trash_keys, on=['title', 'text'])

# 3. Separate the 'Originals' from the 'Copies'
# We keep the MIN index (the first occurrence) for each story
indices_to_keep = all_instances.groupby(['title', 'text'])['index'].min().tolist()
all_indices = all_instances['index'].tolist()
indices_to_delete = list(set(all_indices) - set(indices_to_keep))


initial_total = len(df_cc_news_purged)

# Drop the redundant copies
df_cc_news_purged.drop(index=indices_to_delete, inplace=True)

# THE CRITICAL RESET: Slide the rows up again to fill the gaps
df_cc_news_purged.reset_index(drop=True, inplace=True)

# --- THE REPORT ---
print(f"--- Global Over-40 Purge Complete ---")
print(f"Redundant rows removed: {len(indices_to_delete):,}")
print(f"New 'Golden' total:     {len(df_cc_news_purged):,}")
print(f"Last index check:       {df_cc_news_purged.index[-1]}")

In [ ]:
# 0. RE-GENERATE the summary for the current 701,840 rows
# This ensures we are counting duplicates based on our LATEST dataset
dupe_summary = df_cc_news_purged.groupby(['title', 'text']).size().reset_index(name='count')

# 1. Identify ALL "High-Frequency" stories (> 40 occurrences)
high_freq_stories = dupe_summary[dupe_summary['count'] > 40].copy()

# 2. Map these back to the main dataframe to find their indices
# We use 'df_cc_news_purged' here
high_freq_keys = high_freq_stories[['title', 'text']]
all_high_freq_rows = df_cc_news_purged.reset_index().merge(high_freq_keys, on=['title', 'text'])

# 3. Separate the 'Originals' from the 'Copies'
indices_to_keep = all_high_freq_rows.groupby(['title', 'text'])['index'].min().tolist()
all_indices_in_this_category = all_high_freq_rows['index'].tolist()
indices_to_delete = list(set(all_indices_in_this_category) - set(indices_to_keep))

# --- STATS CAPTURE ---
total_rows_affected = len(all_indices_in_this_category)
unique_stories_kept = len(indices_to_keep)
redundant_copies_deleted = len(indices_to_delete)
initial_total = len(df_cc_news_purged)

# 4. EXECUTE THE PURGE on our golden dataset
df_cc_news_purged.drop(index=indices_to_delete, inplace=True)

# 5. RESET THE INDEX (Sequential Alignment: 0, 1, 2, 3...)
df_cc_news_purged.reset_index(drop=True, inplace=True)

# --- print results ------------
print(f"--- Global 'Over 40' Purge Report ---")
print(f"High-Frequency Groups Found:     {len(high_freq_stories):,}")
print(f"Total High-Frequency Rows:       {total_rows_affected:,}")
print(f"Unique Stories Saved (1 copy):   {unique_stories_kept:,}")
print(f"Redundant Noise Deleted:         {redundant_copies_deleted:,}")
print("-" * 35)
print(f"Dataset Row Count (Before):       {initial_total:,}")
print(f"Dataset Row Count (After):        {len(df_cc_news_purged):,}")
print(f"New Last Index:                  {df_cc_news_purged.index[-1]}")

for the above code we ran a code on initial dataset to delete rows thats is hwy it says 0 right now this was a check

now looking at stories whihc were repeated >=2, and <= 40 times

In [ ]:
# --- STEP 0: INITIAL STATE ---
initial_total = len(df_cc_news_purged)
print(f"--- Starting Residual Purge (2-40) ---")
print(f"Initial Dataset Total: {initial_total:,}")

# 1. Group and count every unique story
story_counts = df_cc_news_purged.groupby(['title', 'text']).size().reset_index(name='repeat_count')

# 2. Filter specifically for the "Residual" range (2 to 40)
residual_targets = story_counts[(story_counts['repeat_count'] >= 2) &
                                (story_counts['repeat_count'] <= 40)].copy()

print(f"Unique Stories Found in 2-40 Range: {len(residual_targets):,}")

# 3. Identify ALL rows in the main dataframe that match these keys
residual_keys = residual_targets[['title', 'text']]
all_residual_rows = df_cc_news_purged.reset_index().merge(residual_keys, on=['title', 'text'])

# 4. Calculate Deletion Plan
indices_to_keep = all_residual_rows.groupby(['title', 'text'])['index'].min().tolist()
all_indices_in_range = all_residual_rows['index'].tolist()
indices_to_delete = list(set(all_indices_in_range) - set(indices_to_keep))

print(f"Redundant Copies Identified to Wipe: {len(indices_to_delete):,}")



In [ ]:
# --- NEW STEP 4.5: VISUAL AUDIT OF TOP 10 STORIES ---
print("\n" + "="*60)
print("VISUAL AUDIT: Top 10 Repeated Stories (Showing up to 5 copies each)")
print("="*60)

top_10_to_audit = residual_targets.head(5)

for i, (idx, row) in enumerate(top_10_to_audit.iterrows(), 1):
    t = row['title']
    tx = row['text']
    count = row['repeat_count']

    # Grab 5 examples from the main dataframe
    examples = df_cc_news_purged[(df_cc_news_purged['title'] == t) & (df_cc_news_purged['text'] == tx)].head(5)

    print(f"\n[STORY #{i}] Found {count} times")
    print(f"TITLE: {t[:100]}")
    print(f"INDICES OF COPIES: {examples.index.tolist()}")
    print(f"TEXT PREVIEW: {tx[:200]}...")
    print("-" * 40)

# --- STOP AND CHECK ---
# If the output above looks correct, proceed to run Step 5 & 6 below

In [ ]:
# Define the clusters of indices you identified
clusters = [
    [156048, 612923],
    [693980, 694239, 695472],
    [21229, 21863]
]

print("--- IPython Style Cluster Inspection ---")

for i, cluster in enumerate(clusters, 1):
    print(f"\nCluster {i}: Indices {cluster}")
    # Displaying using the IPython-friendly head/display format
    display(df_cc_news_purged.loc[cluster, ['title', 'text']])

    # Optional: Check if the text is 100% identical in this cluster
    is_identical = df_cc_news_purged.loc[cluster, 'text'].nunique() == 1
    print(f"Exact Text Match: {is_identical}")
    print("-" * 50)

In [ ]:
# 5. EXECUTE THE PURGE
df_cc_news_purged.drop(index=indices_to_delete, inplace=True)

# 6. FINAL RESET (Ensuring 0 to N-1 sequence)
df_cc_news_purged.reset_index(drop=True, inplace=True)

# --- FINAL ROW CHECK ---
print("-" * 45)
print(f"New 'Golden' Dataset Total: {len(df_cc_news_purged):,}")
print(f"Last Index Check:           {df_cc_news_purged.index[-1]}")
###### print(f"Status: Sequential Integrity Verified.")

jst checking if any stubborn dups are there

In [ ]:
# 1. The Ultimate Duplicate Check (Binary Result)
has_duplicates = df_cc_news_purged.duplicated(subset=['title', 'text']).any()

# 2. Detailed Count of the "Echoes"
total_duplicates = df_cc_news_purged.duplicated(subset=['title', 'text']).sum()

print("--- FINAL DEDUPLICATION AUDIT ---")
if not has_duplicates:
    print("SUCCESS: The dataset is 100% unique. Every row is a distinct row.")
    print(f"Total Rows: {len(df_cc_news_purged):,}")
else:
    print(f" WARNING: {total_duplicates:,} redundant rows were detected!")
    print("We should investigate why these survived.")

# # 3. Quick Title Variance Check
# unique_titles = df_cc_news_purged['title'].nunique()
# print(f"Unique Titles: {unique_titles:,}")
# print(f"Title-to-Row Ratio: {(unique_titles / len(df_cc_news_purged)):.2%}")

Moving to language detection to clean up the dataset for language

fasttext is broken, it didn't work on colab as colab has the latest python, we use google backed package pycld3 for language detectio, it is also on par with other packages., well even glcd3 didn't work, so lingua will be tried untill we find someothing we can actually use


In [ ]:
!pip install lingua-language-detector

In [ ]:
import pandas as pd
from lingua import Language, LanguageDetectorBuilder
from tqdm import tqdm
tqdm.pandas()

looking at only non-english rows

In [ ]:
# 1. Initialize for ALL 75+ languages
# We use .with_low_accuracy_mode() to significantly increase speed
# while maintaining better accuracy than fastText
detector = (LanguageDetectorBuilder.from_all_languages()
            .with_low_accuracy_mode()
            .build())

def is_non_english(text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return False

    # detect_language_of returns a Language object (e.g., Language.ENGLISH)
    detected = detector.detect_language_of(text)

    # We return True if a language was found and it is NOT English
    return detected is not None and detected != Language.ENGLISH

print(f"--- Scanning {len(df_cc_news_purged):,} rows for ANY Non-English content ---")



In [ ]:
# 1. Map the mask to your DataFrame
df_cc_news_purged['is_foreign'] = is_foreign_mask

# 2. Save the entire State to a Pickle file
# This is your 'Save Point' for Monday.
df_cc_news_purged.to_pickle("cc_news_FINAL_AUDIT_MAR25.pkl")

print("SUCCESS: Data is saved to disk. You are safe to close the notebook.")

In [ ]:
total_rows = len(df_cc_news_purged)
foreign_rows = is_foreign_mask.sum()
english_rows = total_rows - foreign_rows

print(f"Total Scanned: {total_rows:,}")
print(f"English (Keep): {english_rows:,} ({(english_rows/total_rows):.2%})")
print(f"Foreign (Drop): {foreign_rows:,} ({(foreign_rows/total_rows):.2%})")

In [ ]:
# Check if the variable 'is_foreign_mask' exists and is full
try:
    print(f"Mask length: {len(is_foreign_mask)}")
    print(f"Sample of results: {is_foreign_mask.head()}")
except NameError:
    print("The variable is gone. It was overwritten.")

In [ ]:
# 2. Apply detection
# This will take longer with 'from_all_languages',

is_foreign_mask = df_cc_news_purged['text'].progress_apply(is_non_english)

# 3. Create the non-English subset
df_non_en = df_cc_news_purged[is_foreign_mask].copy()

# 4. Label the specific languages for the report
if not df_non_en.empty:
    df_non_en['detected_lang'] = df_non_en['text'].apply(
        lambda x: str(detector.detect_language_of(x)).replace("Language.", "")
    )

# # --- THE COMPREHENSIVE REPORT ---
print(f"\n" + "="*10)
print(f"REPORT: GLOBAL NON-ENGLISH SCAN (75+ Languages)")
print(f"="*10)
print(f"Total Foreign Stories: {len(df_non_en):,}")
print(f"Percentage of Dataset:  {(len(df_non_en)/len(df_cc_news_purged)):.2%}")

print("\nTop 15 Languages Detected (Full Dataset):")
if not df_non_en.empty:
    print(df_non_en['detected_lang'].value_counts().head(15))
else:
    print("No non-English content detected.")

In [ ]:
import pandas as pd
import csv
import os

# Define your save directory
save_path = "/content/drive/MyDrive/CC_News_Project/"

# Create the folder if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 1. Save the PICKLE (For Monday's Logic/Speed)
df_cc_news_purged.to_pickle(f"{save_path}cc_news_audit_MAR25.pkl")

# 2. Save the CSV (For your Peace of Mind/Backup)
df_cc_news_purged.to_csv(
    f"{save_path}cc_news_audit_backup_MAR25.csv",
    index=False,
    quoting=csv.QUOTE_ALL,
    encoding='utf-8'
)

print(f"Files successfully banked in Drive at: {save_path}")

**is_foreign = True**,the text was detected as non-English by the Lingua detector → Detector returned a valid language AND it was not English


**is_foreign = False**,
→ The text is treated as English OR not confidently classifiable

As a sanity check I will check the True and False flags - text and title columns

2nd april 2026- From

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/CC_News_Project/cc_news_audit_backup_MAR25.csv'
cc_news = pd.read_csv(file_path)

cc_news.head(5)

sanity checks for is_foreign flag, we will check if the rows which have true as flag are they actually true?


In [ ]:
# distribution and counts of false and true flag
total_rows = len(cc_news)

counts = cc_news['is_foreign'].value_counts(dropna=False)
percent = cc_news['is_foreign'].value_counts(normalize=True, dropna=False) * 100

print(f"Total rows: {total_rows:,}\n")

print("Counts:")
print(counts)

print("\nPercentages:")
print(percent.round(2))

inspect True - flag

In [ ]:
from IPython.display import display
import pandas as pd

# Ensure no truncation
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_columns', None)

# Filter non-English
df_foreign = cc_news[cc_news['is_foreign'] == True]

# Show as full table (no truncation)
display(df_foreign[['title', 'text']].head(20))

before we remove the non-english sentences, we need to bring the data at a normal level, we do these Accent normalization, Newline cleanup, URL removal, Email removal, Lowercasing, Sentence boundary fix, Whitespace normalization -currently I am not removing [., ,,] and apostrophe they are useful for us

REGEX cleaning

In [ ]:
import re
import unicodedata

def clean_text_pipeline(text):
    if not isinstance(text, str):
        return text

    # 1. Remove accents (résumé → resume)
    text = unicodedata.normalize('NFKD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8', 'ignore')

    # 2. Remove newline artifacts
    text = text.replace("/n", "").replace("\n", "").replace("\\n", "")

    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 4. Remove emails
    text = re.sub(r'[A-Za-z0-9+._-]+@[A-Za-z0-9+._-]+\.[A-Za-z0-9+_-]+', '', text)

    # 5. Lowercase
    text = text.lower()

    # 6. Fix sentence boundaries (. ? !)
    # Specifically targets joins like "end.start" -> "end. start"
    text = re.sub(r'([.!?])(?=[a-z])', r'\1 ', text)

    # 7. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
!pip install tqdm

In [ ]:
from tqdm import tqdm
tqdm.pandas()

cc_news['text_clean'] = cc_news['text'].apply(clean_text_pipeline)
cc_news['title_clean'] = cc_news['title'].apply(clean_text_pipeline)

In [ ]:
from IPython.display import display
import pandas as pd

pd.set_option('display.max_colwidth', None)

display(cc_news[['title', 'title_clean', 'text', 'text_clean']].head(2))

In [ ]:
# --- 2. SAVE AS CC_NEWS.CSV ---
# We use 'utf-8-sig' so it opens perfectly in Excel/human-readable software
save_path="/content/drive/MyDrive/CC_News_Project/"
save_file_path = f"{save_path}cc_news.csv"

print(f"Saving purified data to: {save_file_path}")
cc_news.to_csv(save_file_path, index=False, encoding='utf-8-sig')

In [ ]:
cc_news.info()

to some extent we can see it has helped us , now we get onto sentence level cleaning, removing boilerplate phrases etc.

Sentence cleaning

a diff code

This script identifies sentences that appear in hundreds or thousands of different articles. These are your sidebars, navigation menus, and footers.

Boilerplate miner

In [ ]:
from collections import Counter
import re

# We store counts of every sentence found across the corpus
sentence_counts = Counter()

print("Phase 2: Mining the corpus for structural noise...")
for text in tqdm(cc_news['text_clean']):
    # Split by sentence boundaries
    # We only care about sentences between 15 and 200 characters
    sentences = [s.strip() for s in re.split(r'[.!?]', str(text)) if 15 < len(s.strip()) < 200]

    # Use a 'set' per article so we count how many ARTICLES contain the phrase,
    # not how many times it appears in a single article.
    sentence_counts.update(set(sentences))

# Get the top 30 most frequent 'sentences'
global_noise_candidates = sentence_counts.most_common(30)

print("\n--- TOP REPEATED STRUCTURAL NOISE (Potential Blacklist) ---")
for phrase, count in global_noise_candidates:
    print(f"[{count} articles]: {phrase}")

In [ ]:
# Get the top 30 most frequent 'sentences'
global_noise_candidates = sentence_counts.most_common(30)

print("\n--- TOP REPEATED STRUCTURAL NOISE (Potential Blacklist) ---")
for phrase, count in global_noise_candidates:
    print(f"[{count} articles]: {phrase}")

In [ ]:
# 1. Total unique sentences found in the corpus
total_unique = len(sentence_counts)

# 2. Count sentences by frequency buckets
repeats_2_plus = len([s for s, c in sentence_counts.items() if c >= 2])
repeats_10_plus = len([s for s, c in sentence_counts.items() if c >= 10])
repeats_100_plus = len([s for s, c in sentence_counts.items() if c >= 100])
repeats_1000_plus = len([s for s, c in sentence_counts.items() if c >= 1000])

print(f"--- GLOBAL NOISE DISTRIBUTION ---")
print(f"Total Unique Sentences:          {total_unique:,}")
print(f"Repeating in 2+ articles:        {repeats_2_plus:,}  <-- Total 'Noise' Candidates")
print(f"Repeating in 10+ articles:       {repeats_10_plus:,}")
print(f"Repeating in 100+ articles:      {repeats_100_plus:,}  <-- High-Confidence Boilerplate")
print(f"Repeating in 1,000+ articles:    {repeats_1000_plus:,} <-- 'Global' Navigation/Footers")

exporting it to csv

In [ ]:
import pandas as pd

# 1. Convert the Counter object to a list of tuples
# We only take sentences that repeat at least twice (c >= 2)
repeating_data = [
    {"sentence": s, "count": c}
    for s, c in sentence_counts.items() if c >= 2
]

# 2. Create the Audit DataFrame
df_noise_audit = pd.DataFrame(repeating_data)

# 3. Sort by frequency (Highest first)
df_noise_audit = df_noise_audit.sort_values(by="count", ascending=False)
save_path = "/content/drive/MyDrive/CC_News_Project/"
# 4. Save to your Project Folder in Drive
audit_csv_path = f"{save_path}global_noise_audit_APR2.csv"
df_noise_audit.to_csv(audit_csv_path, index=False, encoding='utf-8')

print(f"Audit file saved! You can now open this in Google Sheets or Excel:")
print(f"Path: {audit_csv_path}")
print(f"Total rows to inspect: {len(df_noise_audit):,}")

corpus level cleaning

The Audit JSON: Keeps the counts (so many times the phrase occurs).

The Blacklist JSON: A simple list of strings identified which should be removd

In [ ]:
import json
import pandas as pd

# 1. Create the structured Audit Data (with counts)
# We sort it first so the JSON is organized by highest frequency
repeating_data = [
    {"sentence": s, "count": c}
    for s, c in sentence_counts.items() if c >= 2
]
repeating_data_sorted = sorted(repeating_data, key=lambda x: x['count'], reverse=True)

# 2. Create the flat Blacklist (just the strings for the lookup)
blacklist_only = [item['sentence'] for item in repeating_data_sorted]

# --- FILE PATHS ---
save_path = "/content/drive/MyDrive/CC_News_Project/"
audit_json_path = f"{save_path}global_noise_audit.json"
blacklist_json_path = f"{save_path}global_noise_blacklist_strings.json"

# 3. Save the full Audit JSON
with open(audit_json_path, 'w', encoding='utf-8') as f:
    json.dump(repeating_data_sorted, f, ensure_ascii=False, indent=4)

# 4. Save the lookup-ready Blacklist JSON
with open(blacklist_json_path, 'w', encoding='utf-8') as f:
    json.dump(blacklist_only, f, ensure_ascii=False)

print(f"JSON Files Saved!")
print(f"Audit (with counts): {audit_json_path}")
print(f"Blacklist (strings only): {blacklist_json_path}")

Reloading them -3rd April 2026

just to check the column names

In [ ]:
print(cc_news.columns)

In [ ]:
import json

# Define your paths
audit_path = '/content/drive/MyDrive/CC_News_Project/global_noise_audit.json'
blacklist_path = '/content/drive/MyDrive/CC_News_Project/global_noise_blacklist_strings.json'

# Load the Audit data (usually a dictionary with counts)
with open(audit_path, 'r') as f:
    noise_audit = json.load(f)

# Load the Blacklist (the list of strings for filtering)
with open(blacklist_path, 'r') as f:
    noise_blacklist = json.load(f)

print(f"Successfully reloaded {len(noise_blacklist)} blacklist strings.")

Now that you have the JSON, here is how you use it to "spot and remove" the 850,292 noise phrases. We convert the list to a set immediately because item in set is significantly faster than item in list for a dictionary of this size.

This code  creates a filter for  615,010-row dataset by fragmenting every article into individual sentences and cross-referencing them against your master dictionary of 850,292 repeating phrases. By utilizing a high-speed Python Set, the algorithm instantly spots whether a piece of text is unique news or a recurring "fingerprint" of web noise—such as navigation menus, Hindi footers, or legal boilerplate—that appears in two or more articles across the corpus. it will remvove the repeating junk while carefully rebuilding the original story, ensuring that even fragments without punctuation (like headlines) are preserved, effectively un-scraping the web to leave a purified, high-quality news corpus.

In [ ]:
import json
import re
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
save_path = "/content/drive/MyDrive/CC_News_Project/"
# --- 1. LOAD THE GLOBAL NOISE BLACKLIST ---
# We load the 850k+ strings you identified as Noise
with open(f"{save_path}global_noise_blacklist_strings.json", "r", encoding='utf-8') as f:
    noise_list = json.load(f)

# Convert to a Set for O(1) constant-time lookup (Essential for 615k rows)
noise_lookup = set(noise_list)

# --- 2. REFINED ELIMINATION FUNCTION ---
def eliminate_structural_noise(text, noise_set):
    if not isinstance(text, str) or not text.strip():
        return ""

    # Split by delimiters (. ! ?) while keeping them in the list
    parts = re.split(r'([.!?])', text)

    cleaned_pieces = []

    # We iterate by 2 to grab the sentence body (even index)
    # and check for its punctuation (odd index)
    for i in range(0, len(parts), 2):
        sentence_body = parts[i].strip()

        # Skip empty fragments
        if not sentence_body:
            continue

        # THE LOOKUP: If the sentence is NOT in our 850k noise set, we keep it
        if sentence_body not in noise_set:
            piece = parts[i]
            # Reattach punctuation only if it exists (handles trailing text)
            if i + 1 < len(parts):
                piece += parts[i+1]
            cleaned_pieces.append(piece)

    return "".join(cleaned_pieces).strip()


In [ ]:
# --- 3. APPLY TO DATASET ---
print(f"Purifying 615,010 rows against {len(noise_lookup):,} noise phrases...")

# This overwrites the 'text_clean' column directly
cc_news['text_clean'] = cc_news['text_clean'].progress_apply(lambda x: eliminate_structural_noise(x, noise_lookup))

In [ ]:
cc_news.head(3)

save the file

In [ ]:
cc_news_boilerplt_cleaned = cc_news
save_path = "/content/drive/MyDrive/CC_News_Project/"
final_filename = f"{save_path}cc_news_boilerplt_cleaned.csv"
print(f"Saving your boilerplate cleaned data to {final_filename}...")
cc_news_boilerplt_cleaned.to_csv(final_filename, index=False, encoding='utf-8-sig')

In [ ]:
cc_news_boilerplt_cleaned.head(10)

In [ ]:
cc_news_boilerplt_cleaned  = cc_news_boilerplt_cleaned.drop(columns=["description", "domain"])

8th- calling cc_news_boilerplt_cleaned for further unwanted phrase removal

In [ ]:
import pandas as pd

# Load the file and immediately display the first 2 rows
cc_news_boilerplt_cleaned = pd.read_csv("/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned.csv")
cc_news_boilerplt_cleaned.head(2)

In [ ]:
cc_news_boilerplt_cleaned  = cc_news_boilerplt_cleaned.drop(columns=["description", "domain"])


In [ ]:
cc_news_boilerplt_cleaned.head(2)

Removal of more boilerplate phrases and repeated phrases- using n-gram method , first we look at 20 then we descrease this number and for each n it willbe an iterative process

In [ ]:
# no need to run validation step

In [ ]:
# Validating whether one blacklisted phrase still occurs in the cc_news_boilerplt_cleaned dataset

import pandas as pd

save_path = "/content/drive/MyDrive/CC_News_Project/"
csv_path = save_path + "cc_news_boilerplt_cleaned.csv"

# choose one phrase manually
phrase = "we use cookies to help improve our sites"

text_col = "text_clean"

found = False
total_matches = 0
examples = []

for chunk in pd.read_csv(csv_path, usecols=[text_col], chunksize=5000):
    mask = chunk[text_col].astype(str).str.contains(
        phrase,
        case=False,
        na=False,
        regex=False
    )

    n_matches = mask.sum()

    if n_matches > 0:
        found = True
        total_matches += n_matches

        # store up to 2 example rows
        sample_texts = chunk.loc[mask, text_col].head(2).tolist()
        examples.extend(sample_texts)

        # stop once we already have 2 examples
        if len(examples) >= 2:
            break

print(f"Phrase checked: {phrase}")

if not found:
    print("Number of rows containing it: 0")
    print("Good: this phrase does not appear anymore.")
else:
    print(f"Number of rows containing it so far: {total_matches}")
    print("This phrase is still present.")
    print("\nExample matched rows:")
    for i, text in enumerate(examples[:2], 1):
        print(f"\n--- Match {i} ---")
        print(text)

In [ ]:
# amazing! this news give me so much happiness.

Interative removal of unwanted phrases

N = 20

Step 2 — Run the audit and save the phrase list file

This is the block that actually creates the phrase table we can study.

In [ ]:
!pip install mmh3

In [ ]:
import pandas as pd
import mmh3
import gc
from tqdm.notebook import tqdm
from collections import Counter

def run_positional_audit_ultra_safe(df, n_size, threshold, text_col='text_clean'):
    # --- PASS 1: COUNT HASHES  ---
    # We store NO strings here, only 64-bit numbers
    counter = Counter()
    position_sums = {}

    for text in tqdm(df[text_col], desc="Pass 1: Counting Hashes"):
        lines = str(text).split("\n")
        total_lines = len(lines)
        seen_in_doc = set()

        for i, line in enumerate(lines):
            line = line.strip()
            if not line: continue

            # 0 = Top of doc, 1 = Bottom
            rel_pos = i / (total_lines - 1) if total_lines > 1 else 0
            words = line.split()

            if len(words) >= n_size:
                for j in range(len(words) - n_size + 1):
                    ng = " ".join(words[j:j+n_size])
                    h = mmh3.hash64(ng)[0] # 64-bit integer hash

                    if h not in seen_in_doc:
                        counter[h] += 1
                        position_sums[h] = position_sums.get(h, 0) + rel_pos
                        seen_in_doc.add(h)

    # Identify phrases (Hashes that pass the 100-count mark)
    winners = {h for h, count in counter.items() if count >= threshold}
    print(f"Audit complete. Found {len(winners)} phrases above threshold. Recovering text...")

    # --- PASS 2: RECOVER STRINGS  ---
    # Now we scan again, but ONLY store strings for the few boilerplate phrases
    hash_to_phrase = {}
    for text in tqdm(df[text_col], desc="Pass 2: Recovering Text"):
        if len(hash_to_phrase) == len(winners): break

        lines = str(text).split("\n")
        for line in lines:
            words = line.strip().split()
            if len(words) >= n_size:
                for j in range(len(words) - n_size + 1):
                    ng = " ".join(words[j:j+n_size])
                    h = mmh3.hash64(ng)[0]
                    if h in winners and h not in hash_to_phrase:
                        hash_to_phrase[h] = ng

    # --- FINAL COMPILATION ---
    stats = []
    for h in winners:
        count = counter[h]
        avg_pos = position_sums[h] / count
        stats.append({
            "phrase": hash_to_phrase.get(h, "Unknown"),
            "count": count,
            "avg_pos": round(avg_pos, 3)
        })

    # Force immediate RAM release
    del counter, position_sums, hash_to_phrase, winners
    gc.collect()

    return pd.DataFrame(stats).sort_values(by="count", ascending=False).reset_index(drop=True)

In [ ]:
import os

# Define your save path
save_path = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/ngram20/"

# Create the folder structure
# exist_ok=True prevents an error if the folder already exists
os.makedirs(save_path, exist_ok=True)

16th april, 2026, we started from here.

In [ ]:
# this had the faulty code

24th april, 2026

In [ ]:
# implementing min hash lsh method for discovery of phrases n = 20 across dataset to overcome RAM issues

In [ ]:
!pip install datasketch
!pip install pyarrow

In [ ]:
import os
import gc
import re
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
BASE_DIR            = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR    = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV           = os.path.join(BASE_DIR, "cc_news_boilerplt_cleaned.csv")
SAVE_PATH_PARQUET   = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate.parquet")
SAVE_PATH_CSV       = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate.csv")

# ══════════════════════════════════════════════════════════════════
# 2. CONFIGURATION (Discovery Logic)
# ══════════════════════════════════════════════════════════════════
TEXT_COL         = "text_clean"
N_SIZE           = 20    # Minimum word length to be considered
SIMILARITY       = 0.85  # Jaccard threshold (0.85 = near-exact match)
NUM_PERMUTATIONS = 128   # MinHash resolution
CHUNK_SIZE       = 5000  # Number of rows to process at once to save RAM
THRESHOLD        = 100   # Minimum number of articles phrase must appear in

# ══════════════════════════════════════════════════════════════════
# 3. DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════

def get_shingles(text, n):
    """Breaks text into n-word shingles for hashing."""
    words = str(text).lower().split()
    if len(words) < n: return set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def run_discovery():
    # Initialize the LSH 'Bucket' system
    lsh = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)

    key_to_sentence = {} # Maps bucket keys to the actual text
    key_to_articles = {} # Maps bucket keys to the count of unique articles
    key_counter     = 0

    print(f"Starting Discovery Pass on: {INPUT_CSV}")

    # Pass 1: Streaming through the CSV to find repeats
    # usecols ensures we only load the text column into RAM
    for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE), desc="Discovery"):
        for text in chunk[TEXT_COL].dropna():
            shingles = get_shingles(text, N_SIZE)
            if not shingles: continue

            # Create the 'Fingerprint' (MinHash)
            m = MinHash(num_perm=NUM_PERMUTATIONS)
            for s in shingles: m.update(s.encode('utf8'))

            # Check if this fingerprint is already in a bucket
            neighbors = lsh.query(m)

            if neighbors:
                canonical = neighbors[0]
                key_to_articles[canonical] = key_to_articles.get(canonical, 0) + 1
            else:
                key = f"k{key_counter}"
                lsh.insert(key, m)
                key_to_sentence[key] = text
                key_to_articles[key] = 1
                key_counter += 1

        gc.collect() # Force-clear memory after each chunk

    # ══════════════════════════════════════════════════════════════════
    # 4. HYBRID SAVE (Machine + Human formats)
    # ══════════════════════════════════════════════════════════════════
    boilerplate_data = [
        {"phrase": key_to_sentence[k], "count": count}
        for k, count in key_to_articles.items() if count >= THRESHOLD
    ]

    df_results = pd.DataFrame(boilerplate_data).sort_values("count", ascending=False)

    # Save to Parquet (for the next cleaning step)
    df_results.to_parquet(SAVE_PATH_PARQUET, index=False)

    # Save to CSV (for your manual Excel inspection)
    df_results.to_csv(SAVE_PATH_CSV, index=False)

    print(f"\nPhase 1 Complete!")
    print(f"Discovered Phrases: {len(df_results):,}")
    print(f"Parquet (Machine): {SAVE_PATH_PARQUET}")
    print(f"CSV (Manual Audit): {SAVE_PATH_CSV}")

if __name__ == "__main__":
    run_discovery()

In [ ]:
# looking at the phrase to see if they do indeed turn up in 1841 times or rows as displayed in dataset

import pandas as pd
from IPython.display import display

# 1. Load the dataset
# Ensure INPUT_CSV is defined, e.g.,
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned.csv"
df = pd.read_csv(INPUT_CSV)
TEXT_COL = 'text_clean'

# 2. Define the phrase
target_phrase = "3-nestle takes food price rises in its stridezurich"

# 3. Filter and Create a Copy
# Using .copy() prevents 'SettingWithCopy' warnings when we modify the display columns
matches = df[df[TEXT_COL].str.contains(target_phrase, case=False, na=False)].copy()

# 4. Prepare for Display (Adding Row Numbers)
# We keep the 'index' column to know the exact row in the original CSV
matches = matches.reset_index().rename(columns={'index': 'Original_CSV_Index'})
matches.index = matches.index + 1
matches.index.name = "Row_No"

print(f"Found {len(matches)} rows containing the target phrase.")

# 5. Formatted Table Display
# We use .head(100) to keep the notebook responsive
display(matches.head(100).style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'background-color': '#ffffff',
    'border': '1px solid lightgrey'
}))


In [ ]:
# 6. Export for Inspection
OUTPUT_PATH = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/Boilerphrase_inspection_results1.csv"

# We save the dataframe to a CSV.
# index=True is used here because you set 'Row_No' as the index name above
matches.to_csv(OUTPUT_PATH, index=True)

print(f"Results exported successfully to: {OUTPUT_PATH}")

In [ ]:
# looking at the phrase to see if they do indeed turn up in ~300 times or rows as displayed in dataset

import pandas as pd
from IPython.display import display

# 1. Load the dataset
# Ensure INPUT_CSV is defined, e.g.,
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned.csv"
df1 = pd.read_csv(INPUT_CSV)
TEXT_COL = 'text_clean'

# 2. Define the phrase
target_phrase = "a few things we won't tolerate: personal attacks, obscenity, vulgarity, profanity"

# 3. Filter and Create a Copy
# Using .copy() prevents 'SettingWithCopy' warnings when we modify the display columns
matches1 = df1[df1[TEXT_COL].str.contains(target_phrase, case=False, na=False)].copy()

# 4. Prepare for Display (Adding Row Numbers)
# We keep the 'index' column to know the exact row in the original CSV
matches1 = matches.reset_index().rename(columns={'index': 'Original_CSV_Index'})
matches1.index = matches.index + 1
matches1.index.name = "Row_No"

print(f"Found {len(matches)} rows containing the target phrase.")

# 5. Formatted Table Display
# We use .head(100) to keep the notebook responsive
display(matches1.head(5).style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'background-color': '#ffffff',
    'border': '1px solid lightgrey'
}))

In [ ]:
# 6. Export for Inspection
OUTPUT_PATH = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/Boilerphrase_inspection_results2.csv"

# We save the dataframe to a CSV.
# index=True is used here because you set 'Row_No' as the index name above
matches1.to_csv(OUTPUT_PATH, index=True)

print(f"Results exported successfully to: {OUTPUT_PATH}")

In [ ]:
# checking some more rows to see if I am not deleting relevant rows

In [ ]:
# looking at the phrase to see if they do indeed turn up in ~300 times or rows as displayed in dataset

import pandas as pd
from IPython.display import display

# 1. Load the dataset
# Ensure INPUT_CSV is defined, e.g.,
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned.csv"
df2 = pd.read_csv(INPUT_CSV)
TEXT_COL = 'text_clean'

# 2. Define the phrase
target_phrase = "goldman-backed startup circle launches no-fee foreign payments serviceparis"

# 3. Filter and Create a Copy
# Using .copy() prevents 'SettingWithCopy' warnings when we modify the display columns
matches2 = df2[df2[TEXT_COL].str.contains(target_phrase, case=False, na=False)].copy()

# 4. Prepare for Display (Adding Row Numbers)
# We keep the 'index' column to know the exact row in the original CSV
matches2 = matches2.reset_index().rename(columns={'index': 'Original_CSV_Index'})
matches2.index = matches2.index + 1
matches2.index.name = "Row_No"

print(f"Found {len(matches2)} rows containing the target phrase.")

# 5. Formatted Table Display
# We use .head(100) to keep the notebook responsive
display(matches2.head(5).style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'background-color': '#ffffff',
    'border': '1px solid lightgrey'
}))

I am convinced that these n- 20 min hash lsh phrases that we found and the combination of text and title is just noise and they are not even givving us any important information.

In [ ]:
# I am convinced that these n- 20 min hash lsh phrases that we fpund and the
#combination of text and title is just noise and they are not even givving us any important information


Boiler plate removal part 2

In [ ]:
# removing the rows which are irreleevant to dataset

import pandas as pd

# 1. Load your boilerplate reference and your main dataset
# Replace these paths with your actual file locations
BOILERPLATE_CSV = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/discovered_boilerplate.csv"
MAIN_DATASET_CSV = "/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned.csv"

df_boilerplate = pd.read_csv(BOILERPLATE_CSV)
df_main = pd.read_csv(MAIN_DATASET_CSV)

initial_row_count = len(df_main)
print(f"Initial rows in dataset: {initial_row_count}")

# 2. Get the list of phrases to remove
# We assume the column name in your boilerplate CSV is 'phrase'
phrases_to_remove = df_boilerplate['phrase'].tolist()

# 3. Iteratively remove rows
# We use the tilde (~) operator to keep only rows that do NOT contain the phrase
for i, phrase in enumerate(phrases_to_remove):
    # It is safer to use escape=True or fixed string matching if phrases have special regex characters
    # But for standard cleaning, this is the most direct way:
    df_main = df_main[~df_main['text_clean'].str.contains(phrase, case=False, na=False, regex=False)]

    # Optional: Print progress every 5 phrases so you know it's working
    if (i + 1) % 5 == 0:
        print(f"Processed {i + 1}/{len(phrases_to_remove)} phrases...")

# 4. Final statistics and save
final_row_count = len(df_main)
total_removed = initial_row_count - final_row_count

print(f"\nCleaning Complete!")
print(f"Rows removed: {total_removed}")
print(f"Final dataset size: {final_row_count}")

# Save the newly cleaned version
df_main.to_csv("/content/drive/MyDrive/CC_News_Project/cc_news_boilerplt_cleaned2.csv", index=False)

Boiler plate discovery, inspection and removal- part 3


24TH APRIL 2026 -25H APRIL 2026 RAM CRASHED WE TAKE IT UP FROM HERE

30th april, 2026, we take up discovery, evaluation, removal of repeated phrases n = 15 length

Boilerplate removal part 3

In [ ]:
import os
import gc
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV        = os.path.join(BASE_DIR, "cc_news_boilerplt_cleaned2.csv")
SAVE_PATH_PARQUET = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.parquet")
SAVE_PATH_CSV    = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.csv")

# ══════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ══════════════════════════════════════════════════════════════════
TEXT_COL         = "text_clean"
N_SIZE           = 15    # ← reduced from 20
SIMILARITY       = 0.85
NUM_PERMUTATIONS = 128
CHUNK_SIZE       = 500   # ← reduced from 5000 to control RAM
THRESHOLD        = 10    # ← lower than N=20 run since shorter phrases are rarer per-article

# ══════════════════════════════════════════════════════════════════
# 3. DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════
def get_shingles(text, n):
    """Breaks text into n-word shingles for hashing."""
    words = str(text).lower().split()
    if len(words) < n: return set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def run_discovery():
    lsh             = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)
    key_to_sentence = {}
    key_to_articles = {}
    key_counter     = 0

    print(f"N_SIZE={N_SIZE} | SIMILARITY={SIMILARITY} | THRESHOLD={THRESHOLD} | CHUNK={CHUNK_SIZE}")
    print(f"Input: {INPUT_CSV}\n")

    for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE), desc="Discovery (N=15)"):
        for text in chunk[TEXT_COL].dropna():
            shingles = get_shingles(text, N_SIZE)
            if not shingles: continue

            m = MinHash(num_perm=NUM_PERMUTATIONS)
            for s in shingles: m.update(s.encode('utf8'))

            neighbors = lsh.query(m)

            if neighbors:
                canonical = neighbors[0]
                key_to_articles[canonical] = key_to_articles.get(canonical, 0) + 1
            else:
                key = f"k{key_counter}"
                lsh.insert(key, m)
                key_to_sentence[key] = text
                key_to_articles[key] = 1
                key_counter += 1

        del chunk
        gc.collect()

    # ══════════════════════════════════════════════════════════════════
    # 4. SAVE
    # ══════════════════════════════════════════════════════════════════
    boilerplate_data = [
        {"phrase": key_to_sentence[k], "count": count}
        for k, count in key_to_articles.items() if count >= THRESHOLD
    ]

    df_results = pd.DataFrame(boilerplate_data).sort_values("count", ascending=False)

    df_results.to_parquet(SAVE_PATH_PARQUET, index=False)
    df_results.to_csv(SAVE_PATH_CSV, index=False)

    print(f"\nDiscovery Complete!")
    print(f"Boilerplate phrases found : {len(df_results):,}")
    print(f"Parquet (Machine)         : {SAVE_PATH_PARQUET}")
    print(f"CSV (Manual Audit)        : {SAVE_PATH_CSV}")
    print(f"\nTop 5 preview:")
    print(df_results.head(5).to_string(index=False))

if __name__ == "__main__":
    run_discovery()

In [ ]:
import os
import pandas as pd
from IPython.display import display

INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
SAVE_PATH_CSV = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.csv")

df = pd.read_csv(SAVE_PATH_CSV)

display(df.head(10))

Upon inspection it was found that some phrases which appear in the csv are relevant, but have multiple copies across dataset, some other phrases are just irrelevant, our next task is to identify how we keep the relevant ones and remove the irrelevant ones.

I had to check the phrases and see for which ones I had to keep one copy and for which ones I had to delete


i want to check the rows where action = 1 phrases are being repeated just to check once


In [ ]:
import pandas as pd
import os
from IPython.display import display

BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV        = os.path.join(BASE_DIR, "cc_news_boilerplt_cleaned2.csv")

# 1. Load your manually audited file
# We read the .xlsx you uploaded into the intermediate folder
xlsx_path = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.xlsx")
discovered_boilerplate2 = pd.read_excel(xlsx_path)

# 2. Filter for phrases where you decided to keep one copy (Action 1)
keep_one_phrases = discovered_boilerplate2[discovered_boilerplate2['action'] == 1]['phrase'].unique()

# 3. Read the original large dataset
df_original = pd.read_csv(INPUT_CSV)

# 4. Identify the rows in the original data that contain these 'Action 1' phrases
# NOTE: Replace 'phrase' with the actual column name in your CSV if it's different (e.g., 'text' or 'content')
matches = df_original[df_original['text_clean'].isin(keep_one_phrases)]

# 5. Display in iPython/Colab dataframe style
print(f"Total phrases marked 'Keep One': {len(keep_one_phrases)}")
print(f"Total matching rows found in original dataset: {len(matches)}")

display(matches.head(20))

In [ ]:
import pandas as pd
import os

# 1. Load the audit file
xlsx_path = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.xlsx")
audit_df = pd.read_excel(xlsx_path)

# 2. Pick a phrase from your 'Action 1' list
# Change the index [0] to [1, 2, etc.] to see different repeated stories
action_1_phrases = audit_df[audit_df['action'] == 1]
example_phrase = action_1_phrases.iloc[0]['phrase']

# 3. Load the original dataset (minimal columns for speed)
df_original = pd.read_csv(INPUT_CSV, usecols=['text_clean'])

# 4. Find all rows where this specific phrase is repeated
repeats = df_original[df_original['text_clean'] == example_phrase]

print(f"Phrase being inspected: {example_phrase[:100]}...")
print(f"This phrase appears {len(repeats)} times in your dataset.\n")

# 5. Show the repeating rows
repeats[['text_clean']]

Checked the rows by changing index just to make sure that we do have rows where if we keep one copy then we can delete others.

In [ ]:
import pandas as pd
import os
from IPython.display import display

# 1. Load the audit file
xlsx_path = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.xlsx")
audit_df = pd.read_excel(xlsx_path)

# 2. Grab a phrase from your 'Action 1' (Keep One) list
# You can change [0] to another number to check different repeats
action_1_phrases = audit_df[audit_df['action'] == 1]
target_phrase = action_1_phrases.iloc[2]['phrase']

# 3. Load the original dataset
df_original = pd.read_csv(INPUT_CSV, usecols=['text_clean'])

# 4. Filter for rows that contain the specific phrase
# We use str.contains because the phrase is a snippet of the full text
repeats = df_original[df_original['text_clean'].str.contains(target_phrase, na=False, regex=False)]

print(f"Inspecting phrase: {target_phrase[:100]}...")
print(f"Found {len(repeats)} repeating rows.\n")

# 5. Display in interactive iPython/Colab DataFrame style
display(repeats[['text_clean']])

convinced that we keep first row of different phrases when action = 1, delete the rest as they are repetitions


removal of phrases discovered in n = 15 phase

In [ ]:
import pandas as pd
import os
import gc

BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV        = os.path.join(BASE_DIR, "cc_news_boilerplt_cleaned2.csv")
STEP1_OUTPUT     = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")

# ── Step 1: Load audit file ───────────────────────────────────────────────────
xlsx_path = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_n15.xlsx")
audit_df  = pd.read_excel(xlsx_path)

action_0_phrases = set(audit_df[audit_df['action'] == 0]['phrase'].unique())  # remove all
action_1_phrases = set(audit_df[audit_df['action'] == 1]['phrase'].unique())  # keep first only

print(f"Action 0 phrases (remove all)        : {len(action_0_phrases)}")
print(f"Action 1 phrases (keep first only)   : {len(action_1_phrases)}")

del audit_df
gc.collect()

# ── Step 2: Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
print(f"\nRows loaded: {len(df)}")
rows_before = len(df)

# ── Step 3: Action 0 — Remove ALL occurrences ────────────────────────────────
is_action_0  = df['text_clean'].isin(action_0_phrases)
action_0_removed = is_action_0.sum()

df.drop(index=df.index[is_action_0], inplace=True)
del is_action_0
gc.collect()

print(f"\nAction 0 Cleaning:")
print(f"--- Rows removed (all occurrences)   : {action_0_removed}")
print(f"--- Rows remaining                   : {len(df)}")

# ── Step 4: Action 1 — Keep first, remove duplicates ─────────────────────────
is_action_1      = df['text_clean'].isin(action_1_phrases)
is_duplicate     = df.duplicated(subset=['text_clean'], keep='first')
is_action_1_drop = is_action_1 & is_duplicate
action_1_removed = is_action_1_drop.sum()

del is_action_1, is_duplicate
gc.collect()

df.drop(index=df.index[is_action_1_drop], inplace=True)
del is_action_1_drop
gc.collect()

print(f"\nAction 1 Cleaning:")
print(f"--- Rows removed (duplicates only)   : {action_1_removed}")
print(f"--- Rows remaining                   : {len(df)}")

# ── Step 5: Summary & Save ────────────────────────────────────────────────────
print(f"\nFinal Summary:")
print(f"--- Rows before  : {rows_before}")
print(f"--- Total removed: {rows_before - len(df)}")
print(f"--- Rows after   : {len(df)}")

df.to_csv(STEP1_OUTPUT, index=False)
print(f"\nSaved cleaned file to: {STEP1_OUTPUT}")

completion of removal phase 3


removal phase 4

n = 15 but occurrence more than 2

In [ ]:
import os
import gc
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Updated to use the deduplicated cleaned3 file
INPUT_CSV        = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
SAVE_PATH_PARQUET = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_3.parquet")
SAVE_PATH_CSV    = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate3.csv")

# ══════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ══════════════════════════════════════════════════════════════════
TEXT_COL         = "text_clean"
N_SIZE           = 15    # Target phrase length
SIMILARITY       = 0.85
NUM_PERMUTATIONS = 128
CHUNK_SIZE       = 1000   # Lower chunk size for RAM stability
THRESHOLD        = 2     # Captures any text repeating 2 or more times

# ══════════════════════════════════════════════════════════════════
# 3. DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════
def get_shingles(text, n):
    """Breaks text into n-word shingles for hashing."""
    words = str(text).lower().split()
    if len(words) < n: return set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def run_discovery():
    lsh             = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)
    key_to_sentence = {}
    key_to_articles = {}
    key_counter     = 0

    print(f"N_SIZE={N_SIZE} | SIMILARITY={SIMILARITY} | THRESHOLD={THRESHOLD} | CHUNK={CHUNK_SIZE}")
    print(f"Input: {INPUT_CSV}\n")

    for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE), desc="Discovery (N=15)"):
        for text in chunk[TEXT_COL].dropna():
            shingles = get_shingles(text, N_SIZE)
            if not shingles: continue

            m = MinHash(num_perm=NUM_PERMUTATIONS)
            for s in shingles: m.update(s.encode('utf8'))

            neighbors = lsh.query(m)

            if neighbors:
                canonical = neighbors[0]
                key_to_articles[canonical] = key_to_articles.get(canonical, 0) + 1
            else:
                key = f"k{key_counter}"
                lsh.insert(key, m)
                key_to_sentence[key] = text
                key_to_articles[key] = 1
                key_counter += 1

        del chunk
        gc.collect()

    # ══════════════════════════════════════════════════════════════════
    # 4. SAVE
    # ══════════════════════════════════════════════════════════════════
    # THRESHOLD = 2 filters for occurrences >= 2
    boilerplate_data = [
        {"phrase": key_to_sentence[k], "count": count}
        for k, count in key_to_articles.items() if count >= THRESHOLD
    ]

    df_results = pd.DataFrame(boilerplate_data)

    if not df_results.empty:
        df_results = df_results.sort_values("count", ascending=False)
        df_results.to_parquet(SAVE_PATH_PARQUET, index=False)
        df_results.to_csv(SAVE_PATH_CSV, index=False)

        print(f"\nDiscovery Complete!")
        print(f"Boilerplate clusters found : {len(df_results):,}")
        print(f"Parquet (Machine)          : {SAVE_PATH_PARQUET}")
        print(f"CSV (Manual Audit)         : {SAVE_PATH_CSV}")
        print(f"\nTop 5 preview:")
        print(df_results.head(5).to_string(index=False))
    else:
        print("\nDiscovery Complete! No duplicates found meeting the threshold.")

if __name__ == "__main__":
    run_discovery()

need to check phrases in the dataset

In [ ]:
import os
from IPython.display import display

# --- The Magic UI Fix ---
# This forces the DataFrame to wrap text vertically instead of scrolling horizontally
pd.set_option('display.max_colwidth', None)

# 1. Paths
# BASE_DIR = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
DUPLICATES_CSV = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate3.csv")

# 2. Load the duplicates you just discovered
df_dupes = pd.read_csv(DUPLICATES_CSV)

# 3. Choose which cluster to inspect!
# Change this number (0, 1, 2, 50, 100) to check different duplicate groups
CLUSTER_INDEX = 12084

target_text = df_dupes.iloc[CLUSTER_INDEX]['phrase']
expected_count = df_dupes.iloc[CLUSTER_INDEX]['count']

print(f"--- Inspecting Duplicate Cluster #{CLUSTER_INDEX} ---")
print(f"Expected to find {expected_count} copies.")
print(f"Text Snippet: {str(target_text)[:150]}...\n")

# 4. Load the main dataset (loading 'url' or 'date' helps verify syndication)
# Adjust the usecols if you have other metadata columns you want to see like 'domain' or 'title'
try:
    df_main = pd.read_csv(INPUT_CSV, usecols=['text_clean'])

    # 5. Find all rows in the real dataset that match this text exactly
    matches = df_main[df_main['text_clean'] == target_text]

    print(f"Found {len(matches)} matching rows in the real dataset.")

 # 6. Display the interactive wrapped DataFrame
    display(matches)

except Exception as e:
    print(f"Error loading main dataset: {e}")
    # print("If you don't have a 'url' column, change usecols=['text_clean'] in the code above.")

In [ ]:
import pandas as pd
import os
from IPython.display import display

# Wrap text vertically so it's easy to read
pd.set_option('display.max_colwidth', None)

# 1. Path to your real dataset
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")

# 2. Load the main dataset
df_main = pd.read_csv(INPUT_CSV, usecols=['text_clean'])

# 3. The search snippet you found in Excel
search_snippet = "s. april 16, 2007: virginia tech student seung-hui cho shot"

# 4. Search the MAIN dataset directly for ANY article containing this snippet
matches = df_main[df_main['text_clean'].astype(str).str.contains(search_snippet, na=False, regex=False)]

print(f"Found {len(matches)} highly similar copies in your real dataset.\n")

# 5. Display the interactive wrapped DataFrame
display(matches)

On inspection it was revealed that 0.85% accuracy may have led to not dicovering 100% matches but close matches of the phrases, so before removing any row i had checked whether the last row in the boilerplate 3 csc was actually correct and alidgned with what we get from the cleaned dataset but it was not liek that, we only had one row that contained the last phrase in the boilerplt dataset, so I will modidfy the current code that looks for similar phrases of length 15 words, for ecxact matches, the earlier code where we had mroe than 10 matches was validated so no issues there but for more than 2 or 2 rows we did not get exact copies.


code for n = 15, but more than 2 or 2 rows of exact copies


In [ ]:
import os
import gc
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
# BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Updated to use the deduplicated cleaned3 file
INPUT_CSV        = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
SAVE_PATH_PARQUET = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_4.parquet")
SAVE_PATH_CSV    = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate4.csv")

# ══════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ══════════════════════════════════════════════════════════════════
TEXT_COL         = "text_clean"
N_SIZE           = 15    # Target phrase length
SIMILARITY       = 0.95
NUM_PERMUTATIONS = 128
CHUNK_SIZE       = 1500   # Lower chunk size for RAM stability
THRESHOLD        = 2     # Captures any text repeating 2 or more times

# ══════════════════════════════════════════════════════════════════
# 3. DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════
def get_shingles(text, n):
    """Breaks text into n-word shingles for hashing."""
    words = str(text).lower().split()
    if len(words) < n: return set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def run_discovery():
    lsh             = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)
    key_to_sentence = {}
    key_to_articles = {}
    key_counter     = 0

    print(f"N_SIZE={N_SIZE} | SIMILARITY={SIMILARITY} | THRESHOLD={THRESHOLD} | CHUNK={CHUNK_SIZE}")
    print(f"Input: {INPUT_CSV}\n")

    for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE), desc="Discovery (N=15)"):
        for text in chunk[TEXT_COL].dropna():
            shingles = get_shingles(text, N_SIZE)
            if not shingles: continue

            m = MinHash(num_perm=NUM_PERMUTATIONS)
            for s in shingles: m.update(s.encode('utf8'))

            neighbors = lsh.query(m)

            if neighbors:
                canonical = neighbors[0]
                key_to_articles[canonical] = key_to_articles.get(canonical, 0) + 1
            else:
                key = f"k{key_counter}"
                lsh.insert(key, m)
                key_to_sentence[key] = text
                key_to_articles[key] = 1
                key_counter += 1

        del chunk
        gc.collect()

    # ══════════════════════════════════════════════════════════════════
    # 4. SAVE
    # ══════════════════════════════════════════════════════════════════
    # THRESHOLD = 2 filters for occurrences >= 2
    boilerplate_data = [
        {"phrase": key_to_sentence[k], "count": count}
        for k, count in key_to_articles.items() if count >= THRESHOLD
    ]

    df_results = pd.DataFrame(boilerplate_data)

    if not df_results.empty:
        df_results = df_results.sort_values("count", ascending=False)
        df_results.to_parquet(SAVE_PATH_PARQUET, index=False)
        df_results.to_csv(SAVE_PATH_CSV, index=False)

        print(f"\nDiscovery Complete!")
        print(f"Boilerplate clusters found : {len(df_results):,}")
        print(f"Parquet (Machine)          : {SAVE_PATH_PARQUET}")
        print(f"CSV (Manual Audit)         : {SAVE_PATH_CSV}")
        print(f"\nTop 5 preview:")
        print(df_results.head(5).to_string(index=False))
    else:
        print("\nDiscovery Complete! No duplicates found meeting the threshold.")

if __name__ == "__main__":
    run_discovery()

we start from here tomorrow, we check it and the rows for repetition, then we remove the irrelevant phrases

validating if the phrases are indeed present as many times as its present in the discovered_boilerplate4 csv

1st May 2026


In [ ]:
!pip install pandas

In [ ]:
import os

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
# BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Updated to use the deduplicated cleaned3 file
INPUT_CSV        = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
# SAVE_PATH_PARQUET = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate_4.parquet")
SAVE_PATH_CSV    = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate4.csv")


In [ ]:
import pandas as pd
import sys
import os

# ══════════════════════════════════════════════════════════════════
# 1. YOUR DIRECTORY & PATH SETUP
# ══════════════════════════════════════════════════════════════════
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Input: The full news dataset
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")

# Phrases: The boilerplate discovery list
PHRASES_CSV = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate4.csv")

# Column: The specific column name in your INPUT_CSV that holds the text
PHRASE_COL_NAME = "text_clean"

# Output: Where to save the results
REPORT_PATH = os.path.join(INTERMEDIATE_DIR, "validation_report.csv")

# ── Load ───────────────────────────────────────────────────────────────────────
print("Loading original dataset …")
# Optimization: Only load the specific text column to save RAM
original = pd.read_csv(INPUT_CSV, usecols=[PHRASE_COL_NAME])
print(f"  → {len(original):,} rows loaded.")

print("\nLoading boilerplate phrase list …")
bp = pd.read_csv(PHRASES_CSV, usecols=["phrase", "count"])
bp["count"] = pd.to_numeric(bp["count"], errors="coerce")

# Clean up any non-numeric counts in the boilerplate list
dirty = bp[bp["count"].isna()]
if len(dirty):
    print(f"  ⚠  {len(dirty)} rows with non-numeric count — skipped")

bp = bp.dropna(subset=["count"]).copy()
bp["count"] = bp["count"].astype(int)
print(f"  → {len(bp):,} valid phrase entries")

# ── Validate ───────────────────────────────────────────────────────────────────
print("\nValidating phrase counts … (this may take a minute)")
# This counts occurrences of each phrase in your news dataset
actual_counts = original[PHRASE_COL_NAME].value_counts()

results = []
for _, row in bp.iterrows():
    phrase  = row["phrase"]
    claimed = row["count"]
    actual  = int(actual_counts.get(phrase, 0))
    results.append({
        "phrase":            phrase,
        "claimed_count":     claimed,
        "actual_count":      actual,
        "count_matches":     actual == claimed,
        "discrepancy":       actual - claimed,
        "found_in_original": actual > 0,
        "will_be_removed":   actual >= 2,
        "rows_to_drop":      max(actual - 1, 0),
    })

report = pd.DataFrame(results)

# ── Summary ────────────────────────────────────────────────────────────────────
total           = len(report)
exact_match     = report["count_matches"].sum()
found_not_exact = (report["found_in_original"] & ~report["count_matches"]).sum()
not_found       = (~report["found_in_original"]).sum()
confirmed_dup   = report["will_be_removed"].sum()
rows_to_drop    = report["rows_to_drop"].sum()

print("\n" + "="*58)
print("VALIDATION SUMMARY")
print("="*58)
print(f"  Total phrases checked            : {total:>7,}")
print(f"  ✅ Count matches exactly          : {exact_match:>7,}  ({exact_match/total:.1%})")
print(f"  ⚠  Found but count differs       : {found_not_exact:>7,}  ({found_not_exact/total:.1%})")
print(f"  ❌ Phrase NOT in original at all  : {not_found:>7,}  ({not_found/total:.1%})")
print("="*58)
print(f"\n  Phrases confirmed repeated (≥2x) : {confirmed_dup:>7,}")
print(f"  Duplicate rows that WOULD be dropped : {rows_to_drop:>7,}")
print(f"  Original row count               : {len(original):>7,}")
print(f"  Row count AFTER cleaning (est.)  : {len(original) - rows_to_drop:>7,}")
print("="*58)

# ── Save report ────────────────────────────────────────────────────────────────
report.to_csv(REPORT_PATH, index=False)
print(f"\nValidation report saved → {REPORT_PATH}")

just validating

In [ ]:
import pandas as pd
import os

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & PATH SETUP
# ══════════════════════════════════════════════════════════════════
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
PHRASE_COL_NAME = "text_clean"

# ══════════════════════════════════════════════════════════════════
# 2. PHRASE SELECTION
# ══════════════════════════════════════════════════════════════════
# Verbatim from your search:
LOOKUP_PHRASE = """(cnn) in this terrific new york times piece on donald trump in the white house, one line stood out to me as absolutely critical to understanding how the 45th president of the united states approaches the job. read more"""
# ══════════════════════════════════════════════════════════════════
# 3. EXECUTION
# ══════════════════════════════════════════════════════════════════
print(f"Searching for: '{LOOKUP_PHRASE}'...")

# Load only the columns needed to minimize RAM usage
df_view = pd.read_csv(INPUT_CSV, usecols=[PHRASE_COL_NAME, "title_clean"])

# Filter for the specific boilerplate phrase
matches = df_view[df_view[PHRASE_COL_NAME] == LOOKUP_PHRASE]

print(f"Total occurrences found: {len(matches):,}")

if not matches.empty:
    # Displaying the interactive iPython table
    display(matches.head(50))
else:
    print("❌ No match found. This could be due to hidden characters or extra spaces in the CSV.")

it seems that we need to do validation for step n = 15, 95% similar matches again. we will take it up again tomorrow, from here. 2nd may 2026.

6th may 2026

last time what i found was that there were phrases for which we didnt have phrases appearing as many times as the

In [ ]:
!pip install datasketch

In [ ]:
import os
import gc
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH


In [ ]:
# import os
# import gc
# import pandas as pd
# from tqdm import tqdm
# from datasketch import MinHash, MinHashLSH

# # ══════════════════════════════════════════════════════════════════
# # CONFIGURATION  ← must match your discovery code exactly
# # ══════════════════════════════════════════════════════════════════
# INTERMEDIATE_DIR  = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# INPUT_CSV         = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
# BOILERPLATE_CSV   = os.path.join(INTERMEDIATE_DIR, "discovered_boilerplate4.csv")
# OUTPUT_CSV        = os.path.join(INTERMEDIATE_DIR, "validation_report_v2.csv")

# TEXT_COL          = "text_clean"
# N_SIZE            = 15
# SIMILARITY        = 0.95
# NUM_PERMUTATIONS  = 128
# CHUNK_SIZE        = 1500

# # ══════════════════════════════════════════════════════════════════
# # HELPERS
# # ══════════════════════════════════════════════════════════════════
# def get_minhash(text, n=N_SIZE, num_perm=NUM_PERMUTATIONS):
#     """Identical shingle + hash logic as discovery code."""
#     words = str(text).lower().split()
#     if len(words) < n:
#         return None
#     shingles = {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}
#     m = MinHash(num_perm=num_perm)
#     for s in shingles:
#         m.update(s.encode("utf8"))
#     return m

# # ══════════════════════════════════════════════════════════════════
# # STEP 1 — Build LSH index from boilerplate phrases
# #
# # Each boilerplate phrase gets inserted into an LSH index.
# # Key = its row index (as string), so we can map hits back.
# # This replaces the O(n × m) brute-force pairwise comparison.
# # ══════════════════════════════════════════════════════════════════
# print("=" * 60)
# print("STEP 1: Building LSH index from boilerplate phrases...")
# print("=" * 60)

# bp = pd.read_csv(BOILERPLATE_CSV, usecols=["phrase", "count"])
# bp["count"] = pd.to_numeric(bp["count"], errors="coerce").fillna(0).astype(int)

# lsh         = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)
# bp_minhash  = {}   # index → MinHash object (kept for exact Jaccard check)
# skipped     = 0

# for idx, row in tqdm(bp.iterrows(), total=len(bp), desc="Indexing boilerplate"):
#     m = get_minhash(row["phrase"])
#     if m is None:
#         skipped += 1
#         continue
#     key = str(idx)
#     lsh.insert(key, m)
#     bp_minhash[idx] = m

# print(f"  Indexed : {len(bp_minhash):,} phrases")
# print(f"  Skipped : {skipped} (too short for N={N_SIZE} shingles)\n")

# # ══════════════════════════════════════════════════════════════════
# # STEP 2 — Scan original dataset chunk by chunk
# #
# # For every document in the original CSV:
# #   1. Compute its MinHash  (same N=15 shingle logic)
# #   2. Query the LSH index  → returns candidate boilerplate keys
# #      instantly, without comparing against all 8,467 phrases
# #   3. Verify each candidate with exact Jaccard >= 0.95
# #      (LSH can have false positives, this removes them)
# #   4. Increment actual_count for every confirmed match
# # ══════════════════════════════════════════════════════════════════
# print("=" * 60)
# print("STEP 2: Scanning original dataset...")
# print("=" * 60)

# actual_counts = {idx: 0 for idx in bp_minhash}   # idx → hit count

# for chunk in tqdm(
#     pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE),
#     desc="Scanning dataset"
# ):
#     for text in chunk[TEXT_COL].dropna():
#         m_doc = get_minhash(text)
#         if m_doc is None:
#             continue

#         # LSH query: returns only candidate boilerplate keys (fast)
#         candidates = lsh.query(m_doc)

#         for key in candidates:
#             idx = int(key)
#             # Exact Jaccard check to remove LSH false positives
#             if bp_minhash[idx].jaccard(m_doc) >= SIMILARITY:
#                 actual_counts[idx] += 1

#     del chunk
#     gc.collect()

# # ══════════════════════════════════════════════════════════════════
# # STEP 3 — Build & save validation report
# # ══════════════════════════════════════════════════════════════════
# print("\n" + "=" * 60)
# print("STEP 3: Building validation report...")
# print("=" * 60)

# bp["actual_count"]  = bp.index.map(lambda i: actual_counts.get(i, 0))
# bp["count_matches"] = bp["count"] == bp["actual_count"]
# bp["discrepancy"]   = bp["actual_count"] - bp["count"]

# bp.to_csv(OUTPUT_CSV, index=False)

# # ── Summary ──
# total      = len(bp)
# matched    = bp["count_matches"].sum()
# mismatched = total - matched

# print(f"\n  Total phrases    : {total:,}")
# print(f"  Counts match     : {matched:,}  ({matched/total*100:.1f}%)")
# print(f"  Counts differ    : {mismatched:,}  ({mismatched/total*100:.1f}%)")
# print(f"\n  Saved to: {OUTPUT_CSV}")

# print("\nTop 10 preview:")
# print(bp[["phrase", "count", "actual_count", "count_matches", "discrepancy"]]
#       .head(10)
#       .assign(phrase=bp["phrase"].str[:80])   # truncate for display
#       .to_string(index=False))


validation takes most of the time - it has been 1 hour now, i need to find a way to expedite this will look into a solution for n =10 when we reach that step


broke the csv into parts to be tackled based on what kind of text is there

divided the csv into differnt types of issues, we have 6 different sheets to process, but before that i will check every sheet to see if im not deleting relevant rows and keeping irrelevant ones, dividing the csv was better than validating 8458 phrases at once but yes, Im confident that im not  blindly trusting the code and phrases to remove  relevant phrases

we will find a way to automate this as well

boilerplate 3has those files

we will upload all these files and process each one by one for now, next time we implement block method but for that i need to read about it first we can do that for n =10 more than 100 and more than 2 occurences across dataset

In [ ]:
# Define the directory path
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"

# List all files in the directory and filter for CSVs
# This keeps the names exactly as they appear
files = sorted([f for f in os.listdir(INTERMEDIATE_DIR) if f.endswith('.csv')])

# Dictionary to store dataframes with filenames as keys
dataframes = {}

for file_name in files:
    file_path = os.path.join(INTERMEDIATE_DIR, file_name)
    # Using the filename (without .csv) as the key for easy access
    df_name = file_name.replace(".csv", "")
    dataframes[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file_name} | Shape: {dataframes[df_name].shape}")

# Example: To access the first file specifically
# first_df = dataframes[list(dataframes.keys())[0]]

upon manual inspection i have decided todo the followiing :

wire_service = keep one drop others

encoding artifacts sheet, keep one and drop others, treatment s.s. in the text, numbers, and r.

repeptitive vocab has mix of decision : its a mix of keep one and drop
video_media lost : drop all

repeated chars symb: keep one and drop decided upon inspection
clean_articles
clean-aeticles cscv : mix legit text, okay lets do one thing, lets make a list1. those which are massive dumps of month and year, 2. hacker message, 3. empty rows, 4.row 477, 497, 1593 for these , once we have identified the rows, what we will do is we make a subset of these phrases or rows , we will separate these out, first we will remove these from original dataset, then for the rest of them we separate out and keep one, delete others




clean_articles need special attention

need to come up with strategy for clean_articles.csv

7th may 2026

In [ ]:
import pandas as pd
import os

# --- 1. Configuration and Paths ---
# Directory containing the 6 boilerplate CSVs
boilerplate_dir = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"
# Directory containing the original dataset
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Input and Output file paths
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned3.csv")
WIRE_SERVICE_CSV = os.path.join(boilerplate_dir, "wire_service.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned4.csv")

# IMPORTANT: Change 'text' to the actual name of the column in your original dataset
# that contains the article content (e.g., 'maintext', 'content', etc.)
ORIGINAL_TEXT_COL = 'text_clean'

# --- 2. Load the Data ---
print("Loading original dataset...")
df_original = pd.read_csv(INPUT_CSV)

print(f"Loading boilerplate file: {WIRE_SERVICE_CSV}")
df_wire = pd.read_csv(WIRE_SERVICE_CSV)

# --- 3. Deduplication Logic ---
print("Processing phrases...")

# We will collect the row indices that we want to delete
indices_to_remove = []

# Iterate through every row in the wire_service file
for index, row in df_wire.iterrows():
    phrase_to_find = row['phrase']

    # Locate all rows in the original dataset that match this specific phrase
    # Using exact string matching
    matches = df_original[df_original[ORIGINAL_TEXT_COL] == phrase_to_find]

    count = len(matches)

    if count > 1:
        # We found multiple copies
        # Get the row indices of all matches
        match_indices = matches.index.tolist()

        # Keep the first instance (index 0)
        # Identify everything else (index 1 onwards) for removal
        redundant_copies = match_indices[1:]
        indices_to_remove.extend(redundant_copies)

        print(f"Phrase {index}: Found {count} occurrences. Keeping the first, marking {len(redundant_copies)} for deletion.")

    elif count == 1:
        # If only one copy exists, we do nothing (it stays in the dataset)
        pass

# --- 4. Remove the duplicates and Save ---
# Use set() to ensure we don't try to drop the same index twice
unique_indices_to_remove = list(set(indices_to_remove))

print(f"\nSummary:")
print(f"Total rows before cleaning: {len(df_original)}")
print(f"Total redundant rows to remove: {len(unique_indices_to_remove)}")

# Drop the identified rows
df_cleaned = df_original.drop(index=unique_indices_to_remove)

print(f"Total rows after cleaning: {len(df_cleaned)}")

# Save the final result
df_cleaned.to_csv(OUTPUT_CSV, index=False)
print(f"\nSuccess! Cleaned file saved to: {OUTPUT_CSV}")

next up is

In [ ]:
import pandas as pd
import os

# 1. Define the directory paths
boilerplate_dir = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Input and Output file paths
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned4.csv")
ARTIFACTS_CSV = os.path.join(boilerplate_dir, "encoding_artifacts.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned5.csv")

# 2. Load the original dataset
print(f"Loading original dataset: {os.path.basename(INPUT_CSV)}...")
df_original = pd.read_csv(INPUT_CSV)

# Update this to your actual column name if it is not 'text'
TEXT_COL = 'text_clean'

# 3. Load the encoding artifacts
print(f"Loading artifact phrases from: {os.path.basename(ARTIFACTS_CSV)}...")
df_artifacts = pd.read_csv(ARTIFACTS_CSV)

# This list will store the row indices we want to delete
indices_to_remove = []

# 4. Iterate through every row in the phrase column of encoding_artifacts.csv
print("Starting row-by-row analysis...")
for index, row in df_artifacts.iterrows():
    phrase_to_find = row['phrase']

    # Look for the exact phrase in the original dataset
    # We find all row indices where the text matches the artifact phrase
    matches = df_original[df_original[TEXT_COL] == phrase_to_find].index.tolist()

    count = len(matches)

    if count > 1:
        # DECISION: Keep one copy (matches[0]) and delete other copies
        # We add the "extra" indices (from position 1 onwards) to our removal list
        to_delete = matches[1:]
        indices_to_remove.extend(to_delete)

        # Optional: Print progress for matches
        if index % 100 == 0:
            print(f"Artifact row {index}: Found {count} copies. Keeping 1, removing {len(to_delete)}.")

    elif count == 1:
        # If only one copy exists, we keep that (do nothing)
        pass

# 5. Execute the removal
# Use set() to ensure we don't try to drop the same index twice
unique_indices_to_remove = list(set(indices_to_remove))
df_cleaned = df_original.drop(index=unique_indices_to_remove)

# 6. Final Summary and Save
print("\n" + "="*40)
print(f"Original Row Count: {len(df_original)}")
print(f"Total Rows Removed: {len(unique_indices_to_remove)}")
print(f"Final Row Count:    {len(df_cleaned)}")
print("="*40)

df_cleaned.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved the deduplicated file to: {OUTPUT_CSV}")

next up is

In [ ]:
import pandas as pd
import os

# 1. Define Paths
boilerplate_dir = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned5.csv")
SYMBOL_CSV = os.path.join(boilerplate_dir, "repeated_chars_symbols.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned6.csv")

# Ensure this matches the text column name in your original dataset
TEXT_COL = 'text_clean'

# 2. Load the Data
print(f"Loading original dataset: {os.path.basename(INPUT_CSV)}...")
df_original = pd.read_csv(INPUT_CSV)

print(f"Loading reference file: {os.path.basename(SYMBOL_CSV)}...")
df_symbol = pd.read_csv(SYMBOL_CSV)

# 3. Process Decisions
indices_to_remove = []

print("Processing rows based on 'decision' column...")

for index, row in df_symbol.iterrows():
    phrase = row['phrase']
    decision = str(row['decision']).strip().lower()

    # Find all matches for this phrase in the main dataset
    matches = df_original[df_original[TEXT_COL] == phrase].index.tolist()
    count = len(matches)

    if count > 0:
        if decision == 'keep one':
            # KEEP the first (index 0), mark others [1:] for deletion
            if count > 1:
                to_delete = matches[1:]
                indices_to_remove.extend(to_delete)
                print(f"Row {index} [Keep One]: Found {count} copies. Removing {len(to_delete)} duplicates.")

        elif decision == 'drop':
            # DELETE all occurrences
            indices_to_remove.extend(matches)
            print(f"Row {index} [Drop]: Found {count} copies. Removing all.")

# 4. Execute the Cleaning
# Remove duplicates from the removal list just in case
unique_indices_to_remove = list(set(indices_to_remove))
df_cleaned = df_original.drop(index=unique_indices_to_remove)

# 5. Summary and Save
print("\n" + "="*40)
print(f"Original Row Count: {len(df_original)}")
print(f"Total Rows Removed: {len(unique_indices_to_remove)}")
print(f"Final Row Count:    {len(df_cleaned)}")
print("="*40)

df_cleaned.to_csv(OUTPUT_CSV, index=False)
print(f"Process complete. Saved to: {OUTPUT_CSV}")

wire service, video media list, encoding artifacts, repeated char symbols, now repettitive vocab and clean articles is remaining  now we take up repetititve vocab

In [ ]:
import pandas as pd
import os

# 1. Define Paths
boilerplate_dir = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Input is the result of video_media cleaning (v7)
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned6.csv")
VOCAB_CSV = os.path.join(boilerplate_dir, "repetitive_vocab.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned7.csv")

# Text column name in your original dataset
TEXT_COL = 'text_clean'

# 2. Load the Data
print(f"Loading original dataset: {os.path.basename(INPUT_CSV)}...")
df_original = pd.read_csv(INPUT_CSV)

print(f"Loading vocab reference file: {os.path.basename(VOCAB_CSV)}...")
df_vocab = pd.read_csv(VOCAB_CSV)

# 3. Process Decisions (Keep One vs. Drop)
indices_to_remove = []

print("Processing decisions from repetitive_vocab...")

for index, row in df_vocab.iterrows():
    # Handle potential NaN phrases
    if pd.isna(row['phrase']):
        continue

    phrase = str(row['phrase'])
    decision = str(row['decision']).strip().lower()

    # Find all matches for this phrase in the main dataset
    matches = df_original[df_original[TEXT_COL] == phrase].index.tolist()
    count = len(matches)

    if count > 0:
        if decision == 'keep one':
            # Keep the first match, mark all others for deletion
            if count > 1:
                to_delete = matches[1:]
                indices_to_remove.extend(to_delete)
                print(f"Row {index} [Keep One]: Found {count} copies. Removing {len(to_delete)} duplicates.")

        elif decision == 'drop':
            # Delete every occurrence of this phrase
            indices_to_remove.extend(matches)
            print(f"Row {index} [Drop]: Found {count} copies. Removing all.")

# 4. Execute the Deletion
unique_indices_to_remove = list(set(indices_to_remove))
df_cleaned = df_original.drop(index=unique_indices_to_remove)

# 5. Summary and Save
print("\n" + "="*40)
print(f"Initial Row Count: {len(df_original)}")
print(f"Total Rows Removed: {len(unique_indices_to_remove)}")
print(f"Final Row Count:    {len(df_cleaned)}")
print("="*40)

df_cleaned.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved version 7 to: {OUTPUT_CSV}")

for clean_articles a rule based approach was needed to mark phrases based on the type of issues they had.

In [ ]:
import pandas as pd
import re
import os

# ----------------------------------------------------
# 1. SETUP & LOAD DATA
# ----------------------------------------------------
# Define your specific directory paths
boilerplate_dir = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/boilerplate3"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Define exact file paths
ARTICLES_CSV = os.path.join(boilerplate_dir, "clean_articles.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "final_marked_articles_7thmay.csv")

print(f"Loading data from: {ARTICLES_CSV}")
df = pd.read_csv(ARTICLES_CSV)
df['phrase_str'] = df['phrase'].astype(str)

# Default every row to 'keep one'
df['decision'] = 'keep one'

# ----------------------------------------------------
# 2. APPLY NOISE RULES (Overwrite with 'drop')
# ----------------------------------------------------

# Rule 1: Empty rows or NaNs
empty_mask = df['phrase'].isna() | (df['phrase_str'].str.strip() == '') | (df['phrase_str'] == 'nan')
df.loc[empty_mask, 'decision'] = 'drop - empty'

# Rule 2: Hacker spam
hacker_spam_pattern = r'hacker asked for were a few information|phone could be hacked without having physical access'
hacker_mask = df['phrase_str'].str.contains(hacker_spam_pattern, case=False, na=False)
df.loc[hacker_mask & (df['decision'] == 'keep one'), 'decision'] = 'drop - hacker spam'

# Rule 3: Month Year menus
month_year_pattern = r'(?i)(january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}'
df['month_year_count'] = df['phrase_str'].apply(lambda x: len(re.findall(month_year_pattern, x)))
month_year_mask = df['month_year_count'] > 4
df.loc[month_year_mask & (df['decision'] == 'keep one'), 'decision'] = 'drop - date menu'

# Rule 4: Tag Clouds & Scraped Lists (The Stop Word filter)
stop_words = {'the', 'and', 'to', 'of', 'a', 'in', 'is', 'that', 'for', 'it', 'as', 'was', 'with', 'be', 'by', 'on', 'not', 'he', 'i', 'this', 'are', 'or', 'his', 'from', 'at', 'which', 'but', 'have', 'an', 'had', 'they', 'you', 'were', 'their', 'one', 'all', 'we', 'can', 'her', 'has', 'there', 'been', 'if', 'more', 'when', 'will', 'would', 'who', 'so', 'no', 'about', 'out', 'up', 'said', 'after', 'into', 'over', 'its'}

def calculate_stop_word_ratio(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    if len(words) == 0: return 1.0
    stop_count = sum(1 for w in words if w in stop_words)
    return stop_count / len(words)

df['word_count'] = df['phrase_str'].apply(lambda x: len(re.findall(r'\b[a-z]+\b', x.lower())))
df['stop_word_ratio'] = df['phrase_str'].apply(calculate_stop_word_ratio)

list_mask = (df['word_count'] > 50) & (df['stop_word_ratio'] < 0.15)
df.loc[list_mask & (df['decision'] == 'keep one'), 'decision'] = 'drop - tag cloud'

# ----------------------------------------------------
# 3. CLEANUP AND EXPORT
# ----------------------------------------------------

# Drop the temporary calculation columns
columns_to_drop = ['phrase_str', 'month_year_count', 'word_count', 'stop_word_ratio']
df = df.drop(columns=columns_to_drop)

# See the results
print("\n--- Summary of Decisions ---")
print(df['decision'].value_counts())

# Save the final file to the intermediate directory
print(f"\nSaving marked dataset to: {OUTPUT_CSV}")
df.to_csv(OUTPUT_CSV, index=False)
print("Done!")

upon inspection, i can see that the rules were effective and it has successfully identified the noise text and now i think i can use the final marked artices to clean up the dataset fr n =1 5 and more than 2 ccurences,

In [ ]:
import pandas as pd
import os

# 1. Define Paths
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned7.csv")
MARKED_CSV = os.path.join(INTERMEDIATE_DIR, "final_marked_articles_7thmay.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned8_new.csv")

# NEW: The path for your Decision Log
LOG_CSV = os.path.join(INTERMEDIATE_DIR, "cleanup_decision_log_7thmay.csv")

TEXT_COL = 'text_clean'

# 2. Load the Data
print(f"Loading original dataset: {os.path.basename(INPUT_CSV)}...")
df_original = pd.read_csv(INPUT_CSV)

print(f"Loading marked reference file: {os.path.basename(MARKED_CSV)}...")
df_marked = pd.read_csv(MARKED_CSV)

# 3. Create Instant Lookup Index
print("Building instant lookup index...")
phrase_to_indices = df_original.groupby(TEXT_COL).groups

# 4. Process Decisions & Build the Log
indices_to_remove = []
log_data = [] # This will hold our logs

print("Processing decisions and building the log file in the background...")

for index, row in df_marked.iterrows():
    if pd.isna(row['phrase']):
        continue

    phrase = str(row['phrase'])
    decision = str(row['decision']).strip().lower()

    if phrase in phrase_to_indices:
        matches = list(phrase_to_indices[phrase])
        count = len(matches)

        # Create a tiny snippet of the text just so the log is readable
        text_snippet = phrase[:60] + "..." if len(phrase) > 60 else phrase

        if decision == 'keep one':
            if count > 1:
                to_delete = matches[1:]
                indices_to_remove.extend(to_delete)
                # Log that we found copies and deleted some
                log_data.append({
                    'Checklist_Row': index,
                    'Rule_Applied': 'Keep One',
                    'Total_Copies_Found': count,
                    'Rows_Deleted': len(to_delete),
                    'Text_Snippet': text_snippet
                })
            else:
                # Log that we found it, but it was already unique so we didn't delete anything
                log_data.append({
                    'Checklist_Row': index,
                    'Rule_Applied': 'Keep One',
                    'Total_Copies_Found': count,
                    'Rows_Deleted': 0,
                    'Text_Snippet': text_snippet
                })

        elif decision.startswith('drop'):
            indices_to_remove.extend(matches)
            # Log that we obliterated the bad rows
            log_data.append({
                'Checklist_Row': index,
                'Rule_Applied': decision,
                'Total_Copies_Found': count,
                'Rows_Deleted': count,
                'Text_Snippet': text_snippet
            })

# 5. Execute Deletion
unique_indices_to_remove = list(set(indices_to_remove))
df_cleaned = df_original.drop(index=unique_indices_to_remove)

# 6. Save the Log File
print(f"\nSaving the decision log to: {os.path.basename(LOG_CSV)}...")
df_log = pd.DataFrame(log_data)
df_log.to_csv(LOG_CSV, index=False)

# 7. Summary and Save the Cleaned Data
print("\n" + "="*40)
print(f"Initial Row Count:  {len(df_original)}")
print(f"Total Rows Removed: {len(unique_indices_to_remove)}")
print(f"Final Row Count:    {len(df_cleaned)}")
print("="*40)

print(f"Saving version 8_new to: {OUTPUT_CSV}...")
df_cleaned.to_csv(OUTPUT_CSV, index=False)
print("Successfully completed!")

13th may 2026


before we move on to n = 10 and more than 2 occurences we will clean up the text even more by applying more strategies that we cam eot know about when we started cleaning boilerplate phrases, and also while doing the validation exercise to remvoe the irrelevant phrases and texts

In [ ]:
import os
import pandas as pd
import re

# ==========================================
# 1. Define Paths
# ==========================================
# If you are in Google Colab, make sure you have mounted your drive first:
# from google.colab import drive
# drive.mount('/content/drive')

INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned8_new.csv")
OUTPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned9.csv")
LOG_CSV = os.path.join(INTERMEDIATE_DIR, "cleaning_deletion_log_step9.csv")

# ==========================================
# 2. Define the Cleaning & Logging Function
# ==========================================
deletion_log = []

def clean_and_log(row_id, text):
    if not isinstance(text, str):
        return text

    current_text = text

    # The dictionary of regex rules
    rules = [
        ("Repeating Letters", r'(?i)(?:\b[a-z]\.\s*){2,}', ' '),
        ("Broken Agencies", r'(?i)\b[a-z]\.\s?[a-z]\.\s?(?=\((?:ap|reuters|cnn|source:.*?)\))', ''),
        ("Stray Letters", r'\b[a-z]\.\s+', ' '),
        ("JSON / Ad-Tech Blobs", r'\{[^{}]*?"\w+"\s*:[^{}]*\}', ''),
        ("Aggregated Temporal Data", r'(?i)(?:[a-z]\.)?\s*\d{1,2}:\d{2}(?:\s*[ap]\.?\s*m\.?)?', ' '),
        ("Stock Market Dumps", r'(?i)\b(?:up|down)\s+\$?\d+(?:\.\d+)?\s+to\s+\$?\d+(?:\.\d+)?\.?', ''),
        ("Logic Placeholders", r'\{\*.*?\*\}', '')
    ]

    for rule_name, pattern, replacement in rules:
        # Find exactly what is about to be deleted
        matches = re.findall(pattern, current_text)

        if matches:
            removed_items = [str(m).strip() for m in matches if str(m).strip()]

            if removed_items:
                # Log the deletion event
                deletion_log.append({
                    'Row_ID': row_id,
                    'Rule_Triggered': rule_name,
                    'Removed_Content': " | ".join(removed_items),
                    'Original_Text_Snippet': text[:150] + "..." # First 150 characters for context
                })

            # Execute the deletion
            current_text = re.sub(pattern, replacement, current_text)

    # Clean up double spaces left behind by deletions
    return re.sub(r'\s{2,}', ' ', current_text).strip()

# ==========================================
# 3. Execution Pipeline
# ==========================================
print(f"Loading data from: {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)

# ---> IMPORTANT: CHANGE 'text_column' TO THE ACTUAL NAME OF YOUR COLUMN <---
COLUMN_TO_CLEAN = 'text_clean' # e.g., 'text', 'content', 'article_body'

print(f"Applying cleaning rules to column '{COLUMN_TO_CLEAN}'...")
# Apply the function and track row indices
df[COLUMN_TO_CLEAN] = [clean_and_log(idx, text) for idx, text in df[COLUMN_TO_CLEAN].items()]

print(f"Saving cleaned dataset to: {OUTPUT_CSV}...")
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saving deletion log to: {LOG_CSV}...")
log_df = pd.DataFrame(deletion_log)
log_df.to_csv(LOG_CSV, index=False)

print(f"Complete! Found and removed {len(deletion_log)} errors.")

we recognized other issues as well during boiler plate phrases removal whihc requred this intervention, we need to clean up other issues as well and we will further clean the dataset

ram crash led to changing the code again, new code below

In [ ]:
import os
import pandas as pd
import re
import html
import gc
import shutil

# ==========================================
# 1. Define Paths
# ==========================================
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned9.csv")

#  Write locally first
LOCAL_OUTPUT = "/content/cleaned10_local.csv"
LOCAL_LOG = "/content/log10_local.csv"

#  Final Drive destination
DRIVE_OUTPUT = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned10.csv")
DRIVE_LOG = os.path.join(INTERMEDIATE_DIR, "cleaning_deletion_log_step10.csv")

COLUMN_TO_CLEAN = 'text_clean'

# ==========================================
# 2. Cleaning Function
# ==========================================
def clean_and_log(row_id, text, chunk_log):
    if not isinstance(text, str):
        return text

    current_text = text

    # --- A. HTML Entities ---
    unescaped_text = html.unescape(current_text)
    if unescaped_text != current_text:
        chunk_log.append({
            'Row_ID': row_id,
            'Rule_Triggered': "HTML Entity Decoded",
            'Removed_Content': "Converted raw HTML tags",
            'Original_Text_Snippet': text[:150] + "..."
        })
        current_text = unescaped_text

    # --- B. Mojibake ---
    mojibake_map = {
        'â€"': '-', 'â€"': '-', 'â€™': "'", 'â€œ': '"', 'â€ ': '"'
    }
    for bad_char, good_char in mojibake_map.items():
        if bad_char in current_text:
            chunk_log.append({
                'Row_ID': row_id,
                'Rule_Triggered': "Mojibake Fixed",
                'Removed_Content': f"Replaced '{bad_char}' with '{good_char}'",
                'Original_Text_Snippet': text[:150] + "..."
            })
            current_text = current_text.replace(bad_char, good_char)

    # --- C. Regex Rules ---
    rules = [
        ("Web Boilerplate", r'(?i)\bplease enable javascript to watch this video\.?\b', ''),
        ("Prefix Sluglines", r'(?i)^(?:update\s+\d+-|corrected-|exclusive-|timeline-|factbox-)\s*', '')
    ]
    for rule_name, pattern, replacement in rules:
        matches = re.findall(pattern, current_text)
        if matches:
            removed_items = [str(m).strip() for m in matches if str(m).strip()]
            if removed_items:
                chunk_log.append({
                    'Row_ID': row_id,
                    'Rule_Triggered': rule_name,
                    'Removed_Content': " | ".join(removed_items),
                    'Original_Text_Snippet': text[:150] + "..."
                })
            current_text = re.sub(pattern, replacement, current_text)

    return re.sub(r'\s{2,}', ' ', current_text).strip()

# ==========================================
# 3. Execution Pipeline
# ==========================================
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f" Input file not found: {INPUT_CSV}")

# Clean up old local files
if os.path.exists(LOCAL_OUTPUT): os.remove(LOCAL_OUTPUT)
if os.path.exists(LOCAL_LOG): os.remove(LOCAL_LOG)

file_size_mb = os.path.getsize(INPUT_CSV) / (1024 * 1024)
print(f" Input file size: {file_size_mb:.1f} MB")
print(f" Starting processing...")

chunk_size = 10000
total_rows_written = 0
total_errors_fixed = 0

try:
    for i, chunk in enumerate(pd.read_csv(INPUT_CSV, chunksize=chunk_size)):
        print(f" Processing chunk {i+1} ({len(chunk)} rows)...")

        chunk_log = []

        chunk[COLUMN_TO_CLEAN] = [
            clean_and_log(idx, text, chunk_log)
            for idx, text in chunk[COLUMN_TO_CLEAN].items()
        ]

        # Save chunk to LOCAL disk
        write_mode = 'w' if i == 0 else 'a'
        write_header = (i == 0)
        chunk.to_csv(LOCAL_OUTPUT, mode=write_mode, index=False, header=write_header)
        total_rows_written += len(chunk)

        # Save log to LOCAL disk and clear from RAM
        if chunk_log:
            log_df = pd.DataFrame(chunk_log)
            log_df.to_csv(LOCAL_LOG, mode=write_mode, index=False, header=write_header)
            total_errors_fixed += len(chunk_log)
            del log_df

        print(f" Chunk {i+1} done. Rows so far: {total_rows_written}")

        del chunk, chunk_log
        gc.collect()

except Exception as e:
    import traceback
    print(f"ERROR on chunk {i+1}: {e}")
    traceback.print_exc()

# Verify local file before copying
local_size = os.path.getsize(LOCAL_OUTPUT) / (1024 * 1024)
print(f"\nLocal file confirmed: {local_size:.1f} MB, {total_rows_written} rows")

#  Copy to Drive in one single shot
print(f" Copying to Drive...")
shutil.copy2(LOCAL_OUTPUT, DRIVE_OUTPUT)
shutil.copy2(LOCAL_LOG, DRIVE_LOG)

#  Verify Drive file
drive_size = os.path.getsize(DRIVE_OUTPUT) / (1024 * 1024)
print(f"\nComplete! {total_rows_written} rows written, {total_errors_fixed} errors fixed.")
print(f" Drive output confirmed: {drive_size:.1f} MB")
print(f"   Output → {DRIVE_OUTPUT}")
print(f"   Log    → {DRIVE_LOG}")

This code executes a memory-efficient text-cleaning pipeline that systematically strips out non-English characters, emojis, and all variations of written and numeric dates. By processing the data in small chunks, it prevents RAM crashes while automatically vacuuming up any awkward spaces left behind by the deletions to ensure the text remains neatly formatted.

In [ ]:
import os
import pandas as pd
import re
import gc
import shutil

# ==========================================
# 1. Define Paths
# ==========================================
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned10.csv")

#  Write locally first
LOCAL_OUTPUT = "/content/cleaned11_local.csv"
LOCAL_LOG = "/content/log11_local.csv"

#  Final Drive destination
DRIVE_OUTPUT = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned11.csv")
DRIVE_LOG = os.path.join(INTERMEDIATE_DIR, "cleaning_deletion_log_step11.csv")

COLUMN_TO_CLEAN = 'text_clean'

# ==========================================
# 2. Pre-compile ALL Regex Patterns Once
# ==========================================
months = r'(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)'

REGEX_RULES = [
    ("Non-English Symbols",        re.compile(r'[^\x00-\x7F]+')),
    ("Numeric Date",               re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b')),
    ("ISO Date",                   re.compile(r'\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b')),
    ("Written Date (Month First)", re.compile(fr'(?i)\b{months}\s+\d{{1,2}}(?:st|nd|rd|th)?(?:,)?\s+\d{{4}}\b')),
    ("Written Date (Day First)",   re.compile(fr'(?i)\b\d{{1,2}}(?:st|nd|rd|th)?\s+{months}\s+\d{{4}}\b')),
    ("Written Date (Month Year)",  re.compile(fr'(?i)\b{months}\s+\d{{4}}\b')),
]

# ==========================================
# 3. Cleaning Function
# ==========================================
def clean_and_log_step11(row_id, text, chunk_log):
    if not isinstance(text, str):
        return text

    current_text = text
    original_snippet = text[:150] + "..."

    for rule_name, pattern in REGEX_RULES:
        matches = pattern.findall(current_text)
        if matches:
            removed_items = [str(m).strip() for m in matches if str(m).strip()]
            if removed_items:
                chunk_log.append({
                    'Row_ID': row_id,
                    'Rule_Triggered': rule_name,
                    'Removed_Content': " | ".join(removed_items),
                    'Original_Text_Snippet': original_snippet
                })
            current_text = pattern.sub('', current_text)

    return re.sub(r'\s{2,}', ' ', current_text).strip()

# ==========================================
# 4. Execution Pipeline
# ==========================================
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f" Input file not found: {INPUT_CSV}")

#  Clean up old local files
if os.path.exists(LOCAL_OUTPUT): os.remove(LOCAL_OUTPUT)
if os.path.exists(LOCAL_LOG): os.remove(LOCAL_LOG)

file_size_mb = os.path.getsize(INPUT_CSV) / (1024 * 1024)
print(f" Input file size: {file_size_mb:.1f} MB")
print(f" Starting Step 11: Date & Symbol Removal...")

chunk_size = 50000
total_rows_written = 0
total_errors_fixed = 0

try:
    for i, chunk in enumerate(pd.read_csv(INPUT_CSV, chunksize=chunk_size)):
        print(f"  Processing chunk {i+1} ({len(chunk)} rows)...")

        chunk_log = []

        chunk[COLUMN_TO_CLEAN] = [
            clean_and_log_step11(idx, text, chunk_log)
            for idx, text in chunk[COLUMN_TO_CLEAN].items()
        ]

        #  Save to LOCAL disk
        write_mode = 'w' if i == 0 else 'a'
        write_header = (i == 0)
        chunk.to_csv(LOCAL_OUTPUT, mode=write_mode, index=False, header=write_header)
        total_rows_written += len(chunk)

        # Save log to LOCAL disk and clear from RAM
        if chunk_log:
            log_df = pd.DataFrame(chunk_log)
            log_df.to_csv(LOCAL_LOG, mode=write_mode, index=False, header=write_header)
            total_errors_fixed += len(chunk_log)
            del log_df

        print(f" Chunk {i+1} done. Rows so far: {total_rows_written} | Fixes: {total_errors_fixed}")

        del chunk, chunk_log
        gc.collect()

except Exception as e:
    import traceback
    print(f" ERROR on chunk {i+1}: {e}")
    traceback.print_exc()

# Verify local file
local_size = os.path.getsize(LOCAL_OUTPUT) / (1024 * 1024)
print(f"\n Local file confirmed: {local_size:.1f} MB, {total_rows_written} rows")

#  Copy to Drive in one single shot
print(f" Copying to Drive...")
shutil.copy2(LOCAL_OUTPUT, DRIVE_OUTPUT)
shutil.copy2(LOCAL_LOG, DRIVE_LOG)

# Verify Drive file
drive_size = os.path.getsize(DRIVE_OUTPUT) / (1024 * 1024)
print(f"\n Complete! {total_rows_written} rows written, {total_errors_fixed} errors fixed.")
print(f" Drive output confirmed: {drive_size:.1f} MB")
print(f"   Output → {DRIVE_OUTPUT}")
print(f"   Log    → {DRIVE_LOG}")

14th may 2026:  evaluate the csv logs and everything, then we do the n =10 and 2 more than 2 occurence

This script uses Locality Sensitive Hashing (LSH) to scan massive datasets in memory-efficient chunks, hunting for text blocks that are at least 95% similar to one another. It counts these near-duplicates and exports the full text of any block that appears twice or more into a clean file

In [ ]:
!pip install pandas tqdm datasketch

In [ ]:
import os
import gc
import pandas as pd
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH

# ══════════════════════════════════════════════════════════════════
# 1. DIRECTORY & EXPLICIT PATH SETUP
# ══════════════════════════════════════════════════════════════════
# BASE_DIR         = "/content/drive/MyDrive/CC_News_Project/"
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"

# Updated to use the deduplicated cleaned11 file
INPUT_CSV        = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned11.csv")
# SAVE_PATH_PARQUET = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned12.parquet")
SAVE_PATH_CSV    = os.path.join(INTERMEDIATE_DIR, "boilerplt_cleaned12.csv")

# ══════════════════════════════════════════════════════════════════
# 2. CONFIGURATION
# ══════════════════════════════════════════════════════════════════
TEXT_COL         = "text_clean"
N_SIZE           = 10    # Updated target phrase length
SIMILARITY       = 0.95
NUM_PERMUTATIONS = 128
CHUNK_SIZE       = 1500   # Lower chunk size for RAM stability
THRESHOLD        = 2     # Captures any text repeating 2 or more times

# ══════════════════════════════════════════════════════════════════
# 3. DISCOVERY ENGINE
# ══════════════════════════════════════════════════════════════════
def get_shingles(text, n):
    """Breaks text into n-word shingles for hashing."""
    words = str(text).lower().split()
    if len(words) < n: return set()
    return {" ".join(words[i:i+n]) for i in range(len(words) - n + 1)}

def run_discovery():
    lsh             = MinHashLSH(threshold=SIMILARITY, num_perm=NUM_PERMUTATIONS)
    key_to_sentence = {}
    key_to_articles = {}
    key_counter     = 0

    print(f"N_SIZE={N_SIZE} | SIMILARITY={SIMILARITY} | THRESHOLD={THRESHOLD} | CHUNK={CHUNK_SIZE}")
    print(f"Input: {INPUT_CSV}\n")

    for chunk in tqdm(pd.read_csv(INPUT_CSV, usecols=[TEXT_COL], chunksize=CHUNK_SIZE), desc="Discovery (N=10)"):
        for text in chunk[TEXT_COL].dropna():
            shingles = get_shingles(text, N_SIZE)
            if not shingles: continue

            m = MinHash(num_perm=NUM_PERMUTATIONS)
            for s in shingles: m.update(s.encode('utf8'))

            neighbors = lsh.query(m)

            if neighbors:
                canonical = neighbors[0]
                key_to_articles[canonical] = key_to_articles.get(canonical, 0) + 1
            else:
                key = f"k{key_counter}"
                lsh.insert(key, m)
                key_to_sentence[key] = text
                key_to_articles[key] = 1
                key_counter += 1

        del chunk
        gc.collect()

    # ══════════════════════════════════════════════════════════════════
    # 4. SAVE
    # ══════════════════════════════════════════════════════════════════
    # THRESHOLD = 2 filters for occurrences >= 2
    boilerplate_data = [
        {"phrase": key_to_sentence[k], "count": count}
        for k, count in key_to_articles.items() if count >= THRESHOLD
    ]

    df_results = pd.DataFrame(boilerplate_data)

    if not df_results.empty:
        df_results = df_results.sort_values("count", ascending=False)
        df_results.to_parquet(SAVE_PATH_PARQUET, index=False)
        df_results.to_csv(SAVE_PATH_CSV, index=False)

        print(f"\nDiscovery Complete!")
        print(f"Boilerplate clusters found : {len(df_results):,}")
        print(f"Parquet (Machine)          : {SAVE_PATH_PARQUET}")
        print(f"CSV (Manual Audit)         : {SAVE_PATH_CSV}")
        print(f"\nTop 5 preview:")
        print(df_results.head(5).to_string(index=False))
    else:
        print("\nDiscovery Complete! No duplicates found meeting the threshold.")

if __name__ == "__main__":
    run_discovery()

just a peek view in these phrases row-wise

In [ ]:
import pandas as pd
from IPython.display import display

# ══════════════════════════════════════════════════════════════════
# 1. LOAD THE SOURCE DATA (NOT the results file)
# ══════════════════════════════════════════════════════════════════
# We use cleaned11 because it contains all original columns
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/cc_news_boilerplt_cleaned11.csv"

# Load ONLY the two columns you want to see
df = pd.read_csv(INPUT_CSV, usecols=['title_clean', 'text_clean'])

# ══════════════════════════════════════════════════════════════════
# 2. SEARCH FOR YOUR PHRASE
# ══════════════════════════════════════════════════════════════════
SEARCH_TERM = "28. the glibc 2. guix on android! node. js 10. js 10.7. x)."   # <--- Change this to any phrase you are looking for

# Filter the dataframe
matches = df[df['text_clean'].str.contains(SEARCH_TERM, case=False, na=False)]

# ══════════════════════════════════════════════════════════════════
# 3. DISPLAY FULL ROWS
# ══════════════════════════════════════════════════════════════════
pd.set_option('display.max_colwidth', None) # Don't cut off text

if not matches.empty:
    print(f"✅ Found {len(matches):,} rows containing '{SEARCH_TERM}':\n")
    # Display the rows showing only the columns you requested
    display(matches[['title_clean', 'text_clean']].head(50))
else:
    print(f"❌ Could not find '{SEARCH_TERM}' in the 'text_clean' column.")

In [ ]:
import pandas as pd
from IPython.display import display

# ══════════════════════════════════════════════════════════════════
# 1. LOAD SOURCE DATA
# ══════════════════════════════════════════════════════════════════
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/cc_news_boilerplt_cleaned11.csv"

# Load columns + the index (row number)
df = pd.read_csv(INPUT_CSV, usecols=['title_clean', 'text_clean'])

# ══════════════════════════════════════════════════════════════════
# 2. VALIDATION SEARCH
# ══════════════════════════════════════════════════════════════════
# Copy-paste the phrase from your boilerplate_cleaned12.csv here
# I am using a shorter part of the phrase to catch near-duplicates
SEARCH_PHRASE = "$12 | 3 months.$30 | 6 months"

# We use .str.contains with regex=False to find any row that INCLUDES this phrase
# This captures rows even if they have extra text or slight variations
matches = df[df['text_clean'].str.contains(SEARCH_PHRASE, case=False, na=False, regex=False)].copy()

# Add the original row number as a column
matches['original_row_index'] = matches.index

# ══════════════════════════════════════════════════════════════════
# 3. DISPLAY FOR VALIDATION
# ══════════════════════════════════════════════════════════════════
pd.set_option('display.max_colwidth', None)

if not matches.empty:
    print(f"🔍 SEARCH RESULTS FOR: '{SEARCH_PHRASE}'")
    print(f"✅ Found {len(matches)} rows in the original dataset.")
    print("-" * 50)

    # Show Row Number, Title, and Text
    display(matches[['original_row_index', 'title_clean', 'text_clean']].head(120))
else:
    print(f"❌ Phrase not found. Try searching for a smaller piece of the phrase.")

In [ ]:
import pandas as pd
from IPython.display import display

# ══════════════════════════════════════════════════════════════════
# 1. LOAD SOURCE DATA
# ══════════════════════════════════════════════════════════════════
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/cc_news_boilerplt_cleaned11.csv"

# Load columns + the index (row number)
df = pd.read_csv(INPUT_CSV, usecols=['title_clean', 'text_clean'])

# ══════════════════════════════════════════════════════════════════
# 2. VALIDATION SEARCH
# ══════════════════════════════════════════════════════════════════
# Copy-paste the phrase from your boilerplate_cleaned12.csv here
# I am using a shorter part of the phrase to catch near-duplicates
SEARCH_PHRASE = "what you need to know about the massive oroville dam emergency in california pause video: pasco police looking for suv connected to robbery microbes"

# We use .str.contains with regex=False to find any row that INCLUDES this phrase
# This captures rows even if they have extra text or slight variations
matches = df[df['text_clean'].str.contains(SEARCH_PHRASE, case=False, na=False, regex=False)].copy()

# Add the original row number as a column
matches['original_row_index'] = matches.index

# ══════════════════════════════════════════════════════════════════
# 3. DISPLAY FOR VALIDATION
# ══════════════════════════════════════════════════════════════════
pd.set_option('display.max_colwidth', None)

if not matches.empty:
    print(f"🔍 SEARCH RESULTS FOR: '{SEARCH_PHRASE}'")
    print(f"✅ Found {len(matches)} rows in the original dataset.")
    print("-" * 50)

    # Show Row Number, Title, and Text
    display(matches[['original_row_index', 'title_clean', 'text_clean']].head(120))
else:
    print(f"❌ Phrase not found. Try searching for a smaller piece of the phrase.")

We will try to identify more issues and clean up the cleaned12csv file, we have the report and we will look into how we can track more issues from it. and next time we take up language and word limit as well. we take it up on May22nd,Friday. Need to check the boilerplt12 csv for more issues.

In [ ]:
import pandas as pd
from IPython.display import display

# ══════════════════════════════════════════════════════════════════
# 1. LOAD SOURCE DATA
# ══════════════════════════════════════════════════════════════════
INPUT_CSV = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/cc_news_boilerplt_cleaned11.csv"

# Load columns + the index (row number)
df = pd.read_csv(INPUT_CSV, usecols=['title_clean', 'text_clean'])

# ══════════════════════════════════════════════════════════════════
# 2. VALIDATION SEARCH
# ══════════════════════════════════════════════════════════════════
# Copy-paste the phrase from your boilerplate_cleaned12.csv here
# I am using a shorter part of the phrase to catch near-duplicates
SEARCH_PHRASE = "some wore their dancing shoes"

# We use .str.contains with regex=False to find any row that INCLUDES this phrase
# This captures rows even if they have extra text or slight variations
matches = df[df['text_clean'].str.contains(SEARCH_PHRASE, case=False, na=False, regex=False)].copy()

# Add the original row number as a column
matches['original_row_index'] = matches.index

# ══════════════════════════════════════════════════════════════════
# 3. DISPLAY FOR VALIDATION
# ══════════════════════════════════════════════════════════════════
pd.set_option('display.max_colwidth', None)

if not matches.empty:
    print(f"🔍 SEARCH RESULTS FOR: '{SEARCH_PHRASE}'")
    print(f"✅ Found {len(matches)} rows in the original dataset.")
    print("-" * 50)

    # Show Row Number, Title, and Text
    display(matches[['original_row_index', 'title_clean', 'text_clean']].head(120))
else:
    print(f"❌ Phrase not found. Try searching for a smaller piece of the phrase.")

after we looked at the boilerplt12.csv,

aftr we looked at boilerplt12.csv we noticed that there are still some issues that persist and most of the time it took to figure out how to tackle sliding headlines annd that too of varying sizes amongst other issues, so ihad to loook up i knew what i wanted to look for and i knew that these sliding headlines were of varyign sizes and types, some occurring before or after or in between sometimes in the whole text and sometimes it was just the sliding headline combos that kept repeating
so yeah preparing a code to prep for diffeent kinds of issues took time

My aim for today is to prepare the code block fro all ssues i encoutnere for n -10 more than 2 occurences boiletplt12,csv issues basically and run them tomorrow
its 9 already i need to prep it and make a script we also need to see that my ram doesnt crash and it runs smoothly

i need to work on the remove, inline stripping and remvoing diff noise types, now i am thinking of finalizeing a code script reproducible one no hardcoded values,

23rd may 2026

code for removal, inline stripping, dedup - sliding pipeline, inline stripping, filter, dedup

removal ---- code, need to refine code accorind to the test i want to do first becase there are many opeations and i dont ant to do it wrong, so we are merging inline stripping sliding headlinems and then removal and thne deduping for today

first we doa  trial on 100 rows random rows from the original dataset

In [ ]:
"""
TRACK A — MERGED text_clean CLEANING PIPELINE (TRIAL)
======================================================
Four per-row cleaning stages, ONE ordered pipeline, run on a RANDOM
100-row sample of text_clean. Writes TWO CSVs. Real files untouched.

STAGE ORDER (per row):
  1. SLIDING      — STRIP-ONLY. Truncates repeats / strips boilerplate
                    prefixes. NEVER drops a row itself.
  2. INLINE STRIP — datelines, bylines, share buttons, captions, ©,
                    and the social-media embed signature.
  3. REGEX STRIP  — URLs, paywall, metadata tags, gibberish repeats...
  4. DROP VERDICT — the ONLY stage that drops a row. Judges the
                    fully-cleaned text: empty / code-dump / symbol-dump.

DESIGN PRINCIPLES (all confirmed):
  - Sliding runs FIRST: it needs raw structure to detect repeats.
  - Clean first, drop last: a row only dies if STILL junk after cleaning.
  - Stage 1 is strip-only -> exactly ONE place drops rows (Stage 4).
  - Signal 3 (the bare "pause" rule) has been REMOVED.
  - Signal 5 (n-gram repetition) is DETECTION-ONLY: it tags the row in
    the audit but takes no action; Stage 4 makes the real call.
  - Short remainders after a strip are KEPT and passed to Stage 4.
  - text_clean ONLY. title_clean is a separate effort.
  - Tag-cloud detection is NOT here -> separate Track B.

OUTPUTS:
  1. DECISION AUDIT CSV — every row: stage tags, final verdict, before/after.
  2. CLEANED CSV        — the 100 rows as they would come out.
"""

import re
import os
import pandas as pd
from collections import Counter, defaultdict

# ==========================================================
# SHARED CONFIG
# ==========================================================
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV   = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned11.csv")
AUDIT_CSV   = os.path.join(INTERMEDIATE_DIR, "cc_news_TRACKA_TRIAL_DECISIONS.csv")
CLEANED_CSV = os.path.join(INTERMEDIATE_DIR, "cc_news_TRACKA_TRIAL_100CLEANED.csv")

TEXT_COLUMN = "text_clean"

TRIAL_SAMPLE_SIZE = 100
RANDOM_SEED       = 42

# Stage 1 — sliding
SEQ_PROBE_WORDS   = 10        # Signal 2 detection-probe window length
SEQ_OVERLAP_RATIO = 0.50      # how much of the tail must keep matching
NGRAM_SIZE        = 8         # Signal 5 n-gram length
NGRAM_REPEAT_FLAG = 0.25      # Signal 5 tags the row above this ratio

# cross-doc prefix index
PREFIX_MIN_COUNT  = 3
PREFIX_MIN_WORDS  = 5
PREFIX_SEED_WORDS = 8
PREFIX_CHUNKSIZE  = 10_000

# Stage 4 — drop thresholds
TECH_TOKEN_RATIO_THRESHOLD = 0.60
SYMBOL_DENSITY_THRESHOLD   = 0.25


# ==========================================================
# STAGE 1 — SLIDING  (STRIP-ONLY: never drops)
# ==========================================================
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [p.strip() for p in parts if p.strip()]


def find_exact_repeat_boundary(sentences):
    """Index of the first sentence that is an exact repeat of an earlier one."""
    seen = set()
    for i, s in enumerate(sentences):
        norm = s.lower().strip()
        if norm in seen:
            return i
        seen.add(norm)
    return None


def find_sequence_repeat_boundary(text, window=SEQ_PROBE_WORDS,
                                  overlap_threshold=SEQ_OVERLAP_RATIO):
    """
    Use a `window`-word probe to find a candidate repeat point, then
    measure how far the match ACTUALLY extends from there. The probe
    only LOCATES a candidate; the extension measurement is flexible.
    """
    words = text.lower().split()
    n = len(words)
    if n < window * 3:
        return None
    positions = defaultdict(list)
    for i in range(n - window + 1):
        positions[tuple(words[i:i + window])].append(i)
    best = None
    for plist in positions.values():
        if len(plist) < 2:
            continue
        first_pos, second_pos = plist[0], plist[1]
        if second_pos - first_pos < window:
            continue
        max_overlap = min(n - second_pos, n - first_pos)
        overlap = 0
        for j in range(max_overlap):
            if words[first_pos + j] == words[second_pos + j]:
                overlap += 1
            else:
                break
        remaining = n - second_pos
        ratio = overlap / remaining if remaining > 0 else 0
        if ratio >= overlap_threshold and (best is None or second_pos < best):
            best = second_pos
    return best


def word_idx_to_char_idx(text, word_idx):
    """Token-walk with re.finditer — robust to repeated short words."""
    if word_idx <= 0:
        return 0
    count = 0
    for m in re.finditer(r"\S+", text):
        if count == word_idx:
            return m.start()
        count += 1
    return len(text)


def ngram_repeat_ratio(text, n=NGRAM_SIZE):
    words = text.lower().split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    return sum(c - 1 for c in counts.values() if c > 1) / len(ngrams)


def stage1_sliding(text, prefix_index):
    """
    STRIP-ONLY. Returns (cleaned_text, tags).
    tags is a list of 'sliding:*' strings describing what fired.
    Never drops — the remainder (even if short) is returned as-is.
    """
    if not isinstance(text, str) or not text.strip():
        return text, []

    tags = []
    cleaned = text

    # Signal 1 — exact sentence repeat -> truncate
    sentences = split_sentences(cleaned)
    idx = find_exact_repeat_boundary(sentences)
    if idx is not None:
        cleaned = " ".join(sentences[:idx]).strip()
        tags.append(f"sliding:exact_repeat(idx={idx})")

    # Signal 2 — word-sequence overlap -> truncate
    wb = find_sequence_repeat_boundary(cleaned)
    if wb is not None:
        cleaned = cleaned[:word_idx_to_char_idx(cleaned, wb)].strip()
        tags.append(f"sliding:seq_overlap(word={wb})")

    # Signal 4 — cross-doc boilerplate prefix -> strip prefix
    tl = cleaned.lower()
    for prefix, _ in prefix_index:
        if tl.startswith(prefix):
            cleaned = cleaned[len(prefix):].strip()
            tags.append(f"sliding:prefix({len(prefix.split())}w)")
            break

    # Signal 5 — n-gram repetition -> DETECTION-ONLY (tag, no action)
    ratio = ngram_repeat_ratio(cleaned)
    if ratio > NGRAM_REPEAT_FLAG:
        tags.append(f"sliding:ngram_repeat(ratio={ratio:.2f})")

    return cleaned, tags


# ==========================================================
# STAGE 2 — INLINE STRIP
# ==========================================================
WIRE_DATELINE = re.compile(
    r"(?:^|(?<=[.!?])\s)[a-z][a-z\s,]{0,40}?"
    r"\((?:AP|REUTERS|AFP|UPI|PTI|ANI|IANS|CNS)\)\s*[-–—]*\s*",
    flags=re.IGNORECASE,
)
BYLINE = re.compile(
    r"^by\s+(?:the\s+)?(?:[A-Z][\w\-']*(?:\s+|/)){1,5}"
    r"(?:associated\s*press|reuters|cnn|ap|pti|staff\s*reporter|mailonline)?"
    r"\s*[-–—|,:;]?\s*",
    flags=re.IGNORECASE,
)
# NEW: social-media embed signature, e.g.
#   "a post shared by the shade room (@theshaderoom) on at pdt"
# Trailing date/time text is consumed up to 'pdt'/'pst'/'edt'/etc.,
# which also fixes the 'pdtafter' word-fusion scraping artifact.
SOCIAL_EMBED = re.compile(
    r"a post shared by .{0,60}?\(@[\w.]+\).{0,40}?\b(?:pdt|pst|edt|est|cdt|cst|gmt|utc)\b",
    flags=re.IGNORECASE,
)
SHARE_BUTTONS = re.compile(
    r"\b(?:click to (?:share|email|tweet|print)|share this (?:on|article|story)|share on)\b"
    r"(?:\s+(?:facebook|twitter|reddit|linkedin|email|pinterest|whatsapp|telegram))?",
    flags=re.IGNORECASE,
)
NEWSLETTER = re.compile(
    r"\b(?:subscribe to|sign up for|get our)\s+(?:our\s+|the\s+)?"
    r"(?:newsletter|daily emails?|updates?|alerts?|bulletin)\b",
    flags=re.IGNORECASE,
)
POPUP_ARTIFACTS = re.compile(
    r"\(opens in (?:a )?new window\)|"
    r"could not subscribe,?\s*try again later\.?\s*invalid email\.?|"
    r"please enter a valid email",
    flags=re.IGNORECASE,
)
VIDEO_PLAYER = re.compile(
    r"\bvideo\s*(?:loading|unavailable|buffering)\b|\bclick to play\b|"
    r"\btap to play\b|\bcancel\s+play\s+now\b|\bnow\s+playing\s*:\s*",
    flags=re.IGNORECASE,
)
IMAGE_TAGS = re.compile(
    r"\(\s*(?:image|photo|pic|picture|credit|caption)\s*:\s*[^)]{0,80}\)",
    flags=re.IGNORECASE,
)
PRNEWSWIRE = re.compile(r"/prnewswire(?:-[a-z]+)?/\s*[-–—]*\s*", flags=re.IGNORECASE)
COPYRIGHT  = re.compile(r"©\s*\d{4}|copyright\s*©?\s*\d{4}", flags=re.IGNORECASE)

INLINE_RULES = {
    "wire_dateline": WIRE_DATELINE,
    "byline":        BYLINE,
    "social_embed":  SOCIAL_EMBED,     # NEW
    "share_buttons": SHARE_BUTTONS,
    "newsletter":    NEWSLETTER,
    "popup":         POPUP_ARTIFACTS,
    "video_player":  VIDEO_PLAYER,
    "image_tags":    IMAGE_TAGS,
    "prnewswire":    PRNEWSWIRE,
    "copyright":     COPYRIGHT,
}


def stage2_inline(text):
    """All rules run; every match stripped. Returns (cleaned, tags)."""
    if not isinstance(text, str) or not text.strip():
        return text, []
    t, tags = text, []
    for name, pat in INLINE_RULES.items():
        if pat.search(t):
            tags.append(f"inline:{name}")
            t = pat.sub(" ", t)
    return re.sub(r"\s{2,}", " ", t).strip(), tags


# ==========================================================
# STAGE 3 — REGEX STRIP  (v5 ruleset)
# ==========================================================
REGEX_RULES = {
    "url": r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|org|net|gov|edu|io|co|in)(?:/\S*)?\b",
    "seo_meta": r"\b(?:click here|landed on this page|looking for)\b.{0,30}\b(?:read|find|subscribe|link)\b",
    "paywall": r"(?:\b(?:subscribe|access|premium)\b.{0,20}\$\d+|\$\d+\s*[/|]\s*\d*\s*(?:month|year|week)s?\b)",
    "metadata_leak": r"\[\w+\]|\b\w+\s*:\s*\d+(?:[kKmM]|,\d{3})+\b",
    "short_repeats": r"\b(\w{1,3})\.\s*(?:\1\.\s*){3,}",
    "code_block": r"\{\".*?\"\s*:.*?\}",
    "date_archive": r"(?:\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}\b.{0,15}){3,}",
    "place_names": r"(?:\b(?:gram panchayat|tehsil|mandal|zilla parishad)\b.{0,30}){2,}",
}
COMPILED_REGEX = {n: re.compile(p, flags=re.IGNORECASE) for n, p in REGEX_RULES.items()}


def stage3_regex_strip(text):
    """All rules run; every match stripped. Returns (cleaned, tags)."""
    if not isinstance(text, str) or not text.strip():
        return text, []
    t, tags = text, []
    for name, rx in COMPILED_REGEX.items():
        if rx.search(t):
            tags.append(f"regex:{name}")
            t = rx.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip(), tags


# ==========================================================
# STAGE 4 — DROP VERDICT  (the ONLY stage that drops)
# ==========================================================
def stage4_drop_verdict(text):
    """Judges the fully-cleaned text. Returns list of drop reasons ([] = keep)."""
    if not isinstance(text, str) or not text.strip():
        return ["drop:empty_after_strip"]
    tokens = text.split()
    total = len(tokens)
    if total == 0:
        return ["drop:empty_after_strip"]
    drop = []
    tech = sum(1 for tok in tokens if "." in tok and tok.replace(".", "").isalnum())
    if tech / total > TECH_TOKEN_RATIO_THRESHOLD:
        drop.append("drop:tech_token_ratio")
    symbols = sum(1 for c in text if c.isdigit() or c in ".|%:")
    if len(text) > 0 and symbols / len(text) > SYMBOL_DENSITY_THRESHOLD:
        drop.append("drop:symbol_density")
    return drop


# ==========================================================
# FULL PER-ROW PIPELINE
# ==========================================================
def process_row(text, prefix_index):
    original = text

    t, s1 = stage1_sliding(text, prefix_index)   # strip-only
    t, s2 = stage2_inline(t)
    t, s3 = stage3_regex_strip(t)
    drop_hits = stage4_drop_verdict(t)           # only stage that drops

    tags = s1 + s2 + s3

    if drop_hits:
        return {"action": "drop", "tags": " | ".join(tags + drop_hits),
                "cleaned": "", "orig": original}
    if not isinstance(original, str) or t.strip() != original.strip():
        return {"action": "keep_cleaned", "tags": " | ".join(tags) if tags else "clean",
                "cleaned": t, "orig": original}
    return {"action": "keep_original", "tags": "clean", "cleaned": t, "orig": original}


# ==========================================================
# CROSS-DOC PREFIX INDEX  (built from the FULL dataset)
# ==========================================================
def get_seed(text, n=PREFIX_SEED_WORDS):
    return " ".join(text.lower().split()[:n])


def longest_common_prefix_words(texts):
    if not texts:
        return ""
    wl = [t.lower().split() for t in texts]
    lcp = []
    for i in range(min(len(w) for w in wl)):
        if len({w[i] for w in wl}) == 1:
            lcp.append(wl[0][i])
        else:
            break
    return " ".join(lcp)


def build_prefix_index(filepath, text_col):
    print("Building cross-doc prefix index from the FULL dataset...")
    seed_counts = Counter()
    for chunk in pd.read_csv(filepath, usecols=[text_col],
                             chunksize=PREFIX_CHUNKSIZE, on_bad_lines="skip"):
        for text in chunk[text_col].dropna().tolist():
            if isinstance(text, str) and text.strip():
                seed_counts[get_seed(text)] += 1
    frequent = {s for s, c in seed_counts.items() if c >= PREFIX_MIN_COUNT}

    samples = defaultdict(list)
    for chunk in pd.read_csv(filepath, usecols=[text_col],
                             chunksize=PREFIX_CHUNKSIZE, on_bad_lines="skip"):
        for text in chunk[text_col].dropna().tolist():
            if not isinstance(text, str) or not text.strip():
                continue
            s = get_seed(text)
            if s in frequent and len(samples[s]) < 20:
                samples[s].append(text)

    index, seen = [], set()
    for seed, texts in samples.items():
        lcp = longest_common_prefix_words(texts)
        if len(lcp.split()) >= PREFIX_MIN_WORDS and lcp not in seen:
            seen.add(lcp)
            index.append((lcp, seed_counts[seed]))
    index.sort(key=lambda x: -len(x[0].split()))
    print(f"  Prefix index: {len(index)} boilerplate prefixes.\n")
    return index


# ==========================================================
# GOLDEN TESTS
# ==========================================================
def run_golden_tests():
    print("Running golden tests...")
    fails = 0

    # Stage 1 strip-only: a repeat is truncated but row is NOT dropped here
    c, t = stage1_sliding("The mayor spoke today. The mayor spoke today.", [])
    if not any("exact_repeat" in x for x in t):
        print("  FAIL sliding_repeat ->", t); fails += 1
    else:
        print("  PASS sliding_repeat")

    # Stage 2: byline must NOT eat a normal sentence starting with "By"
    safe, _ = stage2_inline("By Monday, the council had finished its review of the plan.")
    if not safe.lower().startswith("by monday"):
        print("  FAIL byline_safe ->", safe); fails += 1
    else:
        print("  PASS byline_safe")

    # Stage 2: a real byline SHOULD strip
    _, h = stage2_inline("By John Smith - The mayor announced a new policy today.")
    if "inline:byline" not in h:
        print("  FAIL byline_real"); fails += 1
    else:
        print("  PASS byline_real")

    # Stage 2 NEW: social-embed signature must strip
    se, sh = stage2_inline(
        "a post shared by the shade room (@theshaderoom) on at pdt the singer responded.")
    if "inline:social_embed" not in sh:
        print("  FAIL social_embed ->", se, sh); fails += 1
    else:
        print("  PASS social_embed")

    # Stage 3: URL stripped, abbreviations kept
    u, uh = stage3_regex_strip("Read at https://example.com/x news from U.S. U.K. E.U.")
    if "regex:url" not in uh or "U.S." not in u:
        print("  FAIL url_vs_abbrev ->", u, uh); fails += 1
    else:
        print("  PASS url_vs_abbrev")

    # Stage 4: symbol dump dropped
    if not stage4_drop_verdict("12.5% | 33.1% | 8.7% : 99.9% | 2024 | 2025 | 7.7%"):
        print("  FAIL symbol_dump"); fails += 1
    else:
        print("  PASS symbol_dump")

    print(f"Golden tests: {6 - fails}/6 passed.\n")
    if fails:
        raise RuntimeError("Golden tests FAILED — fix stages before running.")


# ==========================================================
# TRIAL RUN
# ==========================================================
def main():
    run_golden_tests()
    prefix_index = build_prefix_index(INPUT_CSV, TEXT_COLUMN)

    print(f"Loading full dataset: {INPUT_CSV}")
    df_full = pd.read_csv(INPUT_CSV)
    print(f"Full dataset: {len(df_full):,} rows.")
    if TEXT_COLUMN not in df_full.columns:
        raise ValueError(f"Column '{TEXT_COLUMN}' not found. Available: {list(df_full.columns)}")

    n = min(TRIAL_SAMPLE_SIZE, len(df_full))
    df = df_full.sample(n=n, random_state=RANDOM_SEED).copy()
    print(f"Random trial sample: {n} rows (seed={RANDOM_SEED}).\n")

    audit_rows, cleaned_rows = [], []
    for idx, row in df.iterrows():
        r = process_row(row[TEXT_COLUMN], prefix_index)
        audit_rows.append({
            "row_index":    idx,
            "final_action": r["action"],
            "stage_tags":   r["tags"],
            "orig_len":     len(str(row[TEXT_COLUMN])),
            "cleaned_len":  len(str(r["cleaned"])),
            "orig_text":    str(r["orig"])[:300],
            "cleaned_text": str(r["cleaned"])[:300],
        })
        if r["action"] != "drop":
            out = row.to_dict()
            out[TEXT_COLUMN] = r["cleaned"]
            cleaned_rows.append(out)

    audit   = pd.DataFrame(audit_rows)
    cleaned = pd.DataFrame(cleaned_rows)

    print("=" * 60)
    print(f"TRACK A TRIAL RESULTS on {n} random rows")
    print("=" * 60)
    for action, cnt in audit["final_action"].value_counts().items():
        print(f"  {action:<16}: {cnt}")

    print("\n  --- stage tags fired ---")
    ts = (audit[audit["stage_tags"] != "clean"]["stage_tags"]
          .str.split(" | ").explode().str.replace(r"\(.*?\)", "", regex=True))
    print(ts.value_counts().to_string() if not ts.empty else "   none")
    print("=" * 60)

    print("\nSAMPLE: dropped rows")
    for _, r in audit[audit["final_action"] == "drop"].head(3).iterrows():
        print(f"  [row {r['row_index']}] {r['stage_tags']}")
        print(f"    {r['orig_text']}\n")

    print("SAMPLE: kept-but-cleaned rows — before vs after")
    for _, r in audit[audit["final_action"] == "keep_cleaned"].head(4).iterrows():
        print(f"  [row {r['row_index']}] {r['stage_tags']}")
        print(f"    before ({r['orig_len']} ch): {r['orig_text']}")
        print(f"    after  ({r['cleaned_len']} ch): {r['cleaned_text']}\n")

    audit.to_csv(AUDIT_CSV, index=False)
    cleaned.to_csv(CLEANED_CSV, index=False)
    print(f"Decision audit ({len(audit)} rows) saved to : {AUDIT_CSV}")
    print(f"Cleaned data  ({len(cleaned)} rows) saved to : {CLEANED_CSV}")
    print("\nTRIAL only — your real data files were NOT modified.")
    print("Note: no tag-cloud handling here — that is the separate Track B.")
    print("Inspect the decision audit, then we build the chunked full-run script.")


if __name__ == "__main__":
    main()


need to refine the removal, inline stripping code as thr trial run shows issues that got left behinf, need to refine the code, added more issues, refining the code to add issue that can be tackled additionally, we ran a trial on 100 rows of data in picked at random from last modified dataset the new code will be run on the last modified dataset. we will run this on the last modificed dataset

I have the full pipeline for some of the rules we run it next time we tackle this, once that is done we do the part b, then part c


31st may, 2026

The input is cc_news_boilerplt_cleaned11.csv — a news corpus that has already gone through some prior boilerplate cleaning — and Track A is the next, more aggressive cleaning pass on top of it.
The core idea: News articles scraped from the web are full of non-article noise — wire agency tags, social media embeds, video player UI artifacts, stitched carousel headlines, repeated boilerplate openings, stock tickers, URLs, copyright lines, etc. the code below systematically finds and removes all of this, row by row, in 5 stages.

In [ ]:
"""
TRACK A — FULL-RUN CLEANING PIPELINE (560k rows)
=================================================
Chunked, memory-flat, saves after every chunk. Processes the WHOLE
dataset. Writes TWO CSVs to the intermediate path.

EXECUTION SEQUENCE:
  1. GOLDEN TESTS  — fake-sentence checks; halt if any rule is broken.
  2. BUILD PREFIX INDEX — stream the corpus, NORMALIZING each row as it
     is read, learn repeated cross-document boilerplate openings.
  3. PROCESS IN CHUNKS — 10,000 rows at a time:
        load chunk -> Stage 0 normalize -> Stage 1 sliding (uses index)
        -> Stage 2 inline -> Stage 3 regex -> Stage 4 drop verdict
        -> APPEND results to the two output CSVs -> next chunk.

OUTPUTS (both in INTERMEDIATE_DIR):
  - DECISION LOG CSV — every row: row_index, final_action, stage_tags,
    orig/cleaned length. The full audit trail.
  - CLEANED CSV      — surviving rows with cleaned text_clean.

NO resume logic: a crash stops the run. Chunks already written stay
safely on disk. Re-running starts fresh (first chunk overwrites).
"""

import re
import os
import time
import pandas as pd
from collections import Counter, defaultdict
from tqdm import tqdm

# ==========================================================
# CONFIG
# ==========================================================
INTERMEDIATE_DIR = "/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning"
INPUT_CSV    = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned11.csv")
DECISION_LOG = os.path.join(INTERMEDIATE_DIR, "boilerplt_decision2.csv")
CLEANED_CSV  = os.path.join(INTERMEDIATE_DIR, "cc_news_boilerplt_cleaned12.csv")

TEXT_COLUMN = "text_clean"
CHUNKSIZE   = 10_000

SEQ_PROBE_WORDS   = 10
SEQ_OVERLAP_RATIO = 0.50
NGRAM_SIZE        = 8
NGRAM_REPEAT_FLAG = 0.25

# The carousel seam token — the irreducible UI vocabulary that marks the
# boundary between the first headline and the stitched carousel tail.
# Visible and editable here, not buried inside a regex.
CAROUSEL_SEAM = "pause"

PREFIX_MIN_COUNT   = 3
PREFIX_MIN_WORDS   = 5
PREFIX_MAX_WORDS   = 25
PREFIX_SEED_WORDS  = 8
PREFIX_SAMPLES_CAP = 20
PREFIX_CHUNKSIZE   = 10_000

TECH_TOKEN_RATIO_THRESHOLD = 0.60
SYMBOL_DENSITY_THRESHOLD   = 0.25

# safety gate: abort if cumulative drop fraction exceeds this
MAX_DROP_FRACTION = 0.50


# ==========================================================
# STAGE 0 — NORMALIZE  (glue-repair -> remove digits -> punct cleanup)
# ==========================================================
_RE_LOWER_UPPER  = re.compile(r"(?<=[a-z])(?=[A-Z])")
_RE_LETTER_DIGIT = re.compile(r"(?<=[A-Za-z])(?=\d)")
_RE_DIGIT_LETTER = re.compile(r"(?<=\d)(?=[A-Za-z])")
_RE_PUNCT_GLUE   = re.compile(r"([.!?:])(?=[A-Za-z])")
_RE_DIGITS       = re.compile(r"\d+")
_RE_LEADING_FRAG = re.compile(r"^\s*[^.!?]{0,30}?\s*[:|]\s*")
_RE_SPACED_COLON = re.compile(r"\s+[:;]\s+")
_RE_SMILEY_ART   = re.compile(r"[:;]\s*[\)\(]")
_RE_DANGLING_DASH= re.compile(r"^\s*[-–—]+\s*|\s+[-–—]+\s+(?=[-–—])")
_RE_PUNCT_RUN    = re.compile(r"[:;,\-–—|]{2,}")
_RE_MULTISPACE   = re.compile(r"\s{2,}")


def stage0_normalize(text):
    """Returns (cleaned_text, tags). Used in BOTH the index build and
    per-row processing, so the index and the rows stay consistent."""
    if not isinstance(text, str) or not text.strip():
        return text, []
    tags = []
    t = text

    before = t
    t = _RE_LOWER_UPPER.sub(" ", t)
    t = _RE_LETTER_DIGIT.sub(" ", t)
    t = _RE_DIGIT_LETTER.sub(" ", t)
    t = _RE_PUNCT_GLUE.sub(r"\1 ", t)
    if t != before:
        tags.append("norm:deglue")

    before = t
    t = _RE_DIGITS.sub(" ", t)
    if t != before:
        tags.append("norm:digits_removed")

    before = t
    t = _RE_LEADING_FRAG.sub("", t)
    t = _RE_SMILEY_ART.sub(" ", t)
    t = _RE_SPACED_COLON.sub(" ", t)
    t = _RE_DANGLING_DASH.sub(" ", t)
    t = _RE_PUNCT_RUN.sub(" ", t)
    if t != before:
        tags.append("norm:punct_cleanup")

    return _RE_MULTISPACE.sub(" ", t).strip(), tags


# ==========================================================
# STAGE 1 — SLIDING  (strip-only; prefix length-guarded)
# ==========================================================
def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", text) if p.strip()]


def find_exact_repeat_boundary(sentences):
    seen = set()
    for i, s in enumerate(sentences):
        norm = s.lower().strip()
        if norm in seen:
            return i
        seen.add(norm)
    return None


def find_sequence_repeat_boundary(text, window=SEQ_PROBE_WORDS,
                                  overlap_threshold=SEQ_OVERLAP_RATIO):
    words = text.lower().split()
    n = len(words)
    if n < window * 3:
        return None
    positions = defaultdict(list)
    for i in range(n - window + 1):
        positions[tuple(words[i:i + window])].append(i)
    best = None
    for plist in positions.values():
        if len(plist) < 2:
            continue
        first_pos, second_pos = plist[0], plist[1]
        if second_pos - first_pos < window:
            continue
        max_overlap = min(n - second_pos, n - first_pos)
        overlap = 0
        for j in range(max_overlap):
            if words[first_pos + j] == words[second_pos + j]:
                overlap += 1
            else:
                break
        remaining = n - second_pos
        ratio = overlap / remaining if remaining > 0 else 0
        if ratio >= overlap_threshold and (best is None or second_pos < best):
            best = second_pos
    return best


def word_idx_to_char_idx(text, word_idx):
    if word_idx <= 0:
        return 0
    count = 0
    for m in re.finditer(r"\S+", text):
        if count == word_idx:
            return m.start()
        count += 1
    return len(text)


def ngram_repeat_ratio(text, n=NGRAM_SIZE):
    words = text.lower().split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    counts = Counter(ngrams)
    return sum(c - 1 for c in counts.values() if c > 1) / len(ngrams)


def stage1_sliding(text, prefix_index):
    if not isinstance(text, str) or not text.strip():
        return text, []
    tags = []
    cleaned = text

    sentences = split_sentences(cleaned)
    idx = find_exact_repeat_boundary(sentences)
    if idx is not None:
        cleaned = " ".join(sentences[:idx]).strip()
        tags.append(f"sliding:exact_repeat(idx={idx})")

    wb = find_sequence_repeat_boundary(cleaned)
    if wb is not None:
        cleaned = cleaned[:word_idx_to_char_idx(cleaned, wb)].strip()
        tags.append(f"sliding:seq_overlap(word={wb})")

    # Signal — sliding-headline carousel. Data shows these rows are
    # "[one headline] <CAROUSEL_SEAM> [run of unrelated stitched
    # headlines]". The seam token marks the boundary; everything from
    # the seam onward is carousel tail -> truncate there, keep before.
    # CAROUSEL_SEAM is a named, visible constant (not buried in regex):
    # it is the irreducible UI vocabulary, not hardcoded content.
    seam = re.search(r"\b" + re.escape(CAROUSEL_SEAM) + r"\b", cleaned, re.IGNORECASE)
    if seam is not None:
        cleaned = cleaned[:seam.start()].strip()
        tags.append("sliding:carousel_truncate")

    tl = cleaned.lower()
    for prefix, _ in prefix_index:
        if tl.startswith(prefix):
            remainder = cleaned[len(prefix):].strip()
            if len(remainder.split()) >= 10:
                cleaned = remainder
                tags.append(f"sliding:prefix({len(prefix.split())}w)")
            break

    ratio = ngram_repeat_ratio(cleaned)
    if ratio > NGRAM_REPEAT_FLAG:
        tags.append(f"sliding:ngram_repeat(ratio={ratio:.2f})")

    return cleaned, tags


# ==========================================================
# STAGE 2 — INLINE STRIP
# ==========================================================
WIRE_DATELINE = re.compile(
    r"(?:^|(?<=[.!?])\s)[a-z][a-z\s,]{0,40}?"
    r"\((?:AP|REUTERS|AFP|UPI|PTI|ANI|IANS|CNS)\)\s*[-–—]*\s*",
    flags=re.IGNORECASE)
BARE_AGENCY = re.compile(
    r"^\s*\((?:AP|REUTERS|AFP|UPI|PTI|ANI|IANS|CNS|CNN)\)\s*[-–—]*\s*",
    flags=re.IGNORECASE)
AGENCY_SIGNOFF = re.compile(r"\s*[-–—]{1,2}[a-z]{2,}(?:/[a-z]{2,}){1,4}\s*$",
                            flags=re.IGNORECASE)
BYLINE = re.compile(
    r"^by\s+(?:the\s+)?(?:[A-Z][\w\-']*(?:\s+|/)){1,5}"
    r"(?:associated\s*press|reuters|cnn|ap|pti|staff\s*reporter|mailonline)?"
    r"\s*[-–—|,:;]?\s*",
    flags=re.IGNORECASE)
LEFTOVER_NAME_FRAG = re.compile(r"^\s*[a-z][a-z\-']{1,20}\s*:\s*", flags=re.IGNORECASE)
SOCIAL_EMBED = re.compile(
    r"a post shared by .{0,60}?\(@[\w.]+\).{0,40}?\b(?:pdt|pst|edt|est|cdt|cst|gmt|utc)\b",
    flags=re.IGNORECASE)
BARE_HANDLE = re.compile(r"(?<![\w(])@[A-Za-z][\w]{2,}")
SPACED_PIC_URL = re.compile(r"pic\.\s*twitter\.\s*com\S*", flags=re.IGNORECASE)
SHARE_BUTTONS = re.compile(
    r"\b(?:click to (?:share|email|tweet|print)|share this (?:on|article|story)|share on)\b"
    r"(?:\s+(?:facebook|twitter|reddit|linkedin|email|pinterest|whatsapp|telegram))?",
    flags=re.IGNORECASE)
NEWSLETTER = re.compile(
    r"\b(?:subscribe to|sign up for|get our)\s+(?:our\s+|the\s+)?"
    r"(?:newsletter|daily emails?|updates?|alerts?|bulletin)\b",
    flags=re.IGNORECASE)
POPUP_ARTIFACTS = re.compile(
    r"\(opens in (?:a )?new window\)|"
    r"could not subscribe,?\s*try again later\.?\s*invalid email\.?|"
    r"please enter a valid email",
    flags=re.IGNORECASE)
GALLERY_VIDEO = re.compile(
    r"\bskip in skip\b.*?\bshare\b|\bembed x share\b|\bimage \w+ of\s*/?\s*\w+\b"
    r"|\bcaption close\b|\b\w{2,12}\svideo\s*:|\bvideo\s*(?:loading|unavailable|buffering)\b"
    r"|\bclick to play\b|\btap to play\b|\bnow\s+playing\s*:\s*",
    flags=re.IGNORECASE)
IMAGE_TAGS = re.compile(
    r"\(\s*(?:image|photo|pic|picture|credit|caption)\s*:\s*[^)]{0,80}\)",
    flags=re.IGNORECASE)
PHOTO_CREDIT = re.compile(
    r"\b(?:enlarge this image\s*)?toggle caption\b.{0,80}?\b(?:ap|reuters|afp|getty)\b",
    flags=re.IGNORECASE)
EQ_DIVIDER = re.compile(r"={4,}")
PRNEWSWIRE = re.compile(r"/prnewswire(?:-[a-z]+)?/\s*[-–—]*\s*", flags=re.IGNORECASE)
COPYRIGHT  = re.compile(r"©\s*\d{0,4}|copyright\s*©?\s*\d{0,4}", flags=re.IGNORECASE)

INLINE_RULES = {
    "wire_dateline":  WIRE_DATELINE,
    "bare_agency":    BARE_AGENCY,
    "agency_signoff": AGENCY_SIGNOFF,
    "byline":         BYLINE,
    "leftover_frag":  LEFTOVER_NAME_FRAG,
    "social_embed":   SOCIAL_EMBED,
    "bare_handle":    BARE_HANDLE,
    "spaced_pic_url": SPACED_PIC_URL,
    "share_buttons":  SHARE_BUTTONS,
    "newsletter":     NEWSLETTER,
    "popup":          POPUP_ARTIFACTS,
    "gallery_video":  GALLERY_VIDEO,
    "image_tags":     IMAGE_TAGS,
    "photo_credit":   PHOTO_CREDIT,
    "eq_divider":     EQ_DIVIDER,
    "prnewswire":     PRNEWSWIRE,
    "copyright":      COPYRIGHT,
}


def stage2_inline(text):
    if not isinstance(text, str) or not text.strip():
        return text, []
    t, tags = text, []
    for name, pat in INLINE_RULES.items():
        if pat.search(t):
            tags.append(f"inline:{name}")
            t = pat.sub(" ", t)
    return re.sub(r"\s{2,}", " ", t).strip(), tags


# ==========================================================
# STAGE 3 — REGEX STRIP
# (date_archive REMOVED: digits + month-name matching no longer
#  meaningful after Stage 0 strips all digits.)
# ==========================================================
REGEX_RULES = {
    # obfuscated_email MUST run before url (url would eat the domain).
    "obfuscated_email": r"\b[\w.\-]*\*{2,}[\w.\-]*@[\w.\-]+",
    "stock_ticker": r"\(\s*(?:nasdaq|nyse|bse|nse|lse|otc)\s*:\s*[^)]{1,12}\)",
    "url": r"(?:https?://|www\.)\S+|\b[\w-]+\.(?:com|org|net|gov|edu|io|co|in)(?:/\S*)?\b",
    "seo_meta": r"\b(?:click here|landed on this page|looking for)\b.{0,30}\b(?:read|find|subscribe|link)\b",
    "paywall": r"(?:\b(?:subscribe|access|premium)\b.{0,20}\$\d+|\$\d+\s*[/|]\s*\d*\s*(?:month|year|week)s?\b)",
    "metadata_leak": r"\[\w+\]|\b\w+\s*:\s*\d+(?:[kKmM]|,\d{3})+\b",
    "short_repeats": r"\b(\w{1,3})\.\s*(?:\1\.\s*){3,}",
    "code_block": r"\{[^{}]*\"[^{}]*:[^{}]*\}|[\"']\w+[\"']\s*[:}]",
    "place_names": r"(?:\b(?:gram panchayat|tehsil|mandal|zilla parishad)\b.{0,30}){2,}",
}
COMPILED_REGEX = {n: re.compile(p, flags=re.IGNORECASE) for n, p in REGEX_RULES.items()}


def stage3_regex_strip(text):
    if not isinstance(text, str) or not text.strip():
        return text, []
    t, tags = text, []
    for name, rx in COMPILED_REGEX.items():
        if rx.search(t):
            tags.append(f"regex:{name}")
            t = rx.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip(), tags


# ==========================================================
# STAGE 4 — DROP VERDICT  (the ONLY stage that drops)
# ==========================================================
# Repeated-schedule rows. After Stage 0 strips digits, a schedule like
# "players 161-165 (5/28)players 156-160..." becomes the SAME short token
# repeated 4+ times, each followed by bracket/slash/dash punctuation.
# Structural — the \1 backreference keys on "a token repeating", not on
# any specific word like "players".
RELEASE_SCHED = re.compile(
    r"release dates?\b|\b(\w{3,12})\b[^a-z]{1,20}(?:\b\1\b[^a-z]{1,20}){3,}",
    flags=re.IGNORECASE)


def stage4_drop_verdict(text):
    if not isinstance(text, str) or not text.strip():
        return ["drop:empty_after_strip"]
    tokens = text.split()
    total = len(tokens)
    if total == 0:
        return ["drop:empty_after_strip"]
    drop = []
    if RELEASE_SCHED.search(text):
        drop.append("drop:release_schedule")
    tech = sum(1 for tok in tokens if "." in tok and tok.replace(".", "").isalnum())
    if tech / total > TECH_TOKEN_RATIO_THRESHOLD:
        drop.append("drop:tech_token_ratio")
    symbols = sum(1 for c in text if c.isdigit() or c in ".|%:")
    if len(text) > 0 and symbols / len(text) > SYMBOL_DENSITY_THRESHOLD:
        drop.append("drop:symbol_density")
    return drop


# ==========================================================
# FULL PER-ROW PIPELINE
# ==========================================================
def process_row(text, prefix_index):
    original = text
    t, s0 = stage0_normalize(text)
    t, s1 = stage1_sliding(t, prefix_index)
    t, s2 = stage2_inline(t)
    t, s3 = stage3_regex_strip(t)
    drop_hits = stage4_drop_verdict(t)
    tags = s0 + s1 + s2 + s3

    if drop_hits:
        return {"action": "drop", "tags": " | ".join(tags + drop_hits), "cleaned": ""}
    if not isinstance(original, str) or t.strip() != original.strip():
        return {"action": "keep_cleaned",
                "tags": " | ".join(tags) if tags else "clean", "cleaned": t}
    return {"action": "keep_original", "tags": "clean", "cleaned": t}


# ==========================================================
# CROSS-DOC PREFIX INDEX  (normalization applied INSIDE the build)
# ==========================================================
def get_seed(text, n=PREFIX_SEED_WORDS):
    return " ".join(text.lower().split()[:n])


def longest_common_prefix_words(texts):
    if not texts:
        return ""
    wl = [t.lower().split() for t in texts]
    lcp = []
    for i in range(min(len(w) for w in wl)):
        if len({w[i] for w in wl}) == 1:
            lcp.append(wl[0][i])
        else:
            break
    return " ".join(lcp)


def build_prefix_index(filepath, text_col):
    print("Building cross-doc prefix index (normalizing rows during build)...")

    # Pass 1 — count seeds of NORMALIZED text
    seed_counts = Counter()
    for chunk in tqdm(pd.read_csv(filepath, usecols=[text_col],
                                  chunksize=PREFIX_CHUNKSIZE, on_bad_lines="skip"),
                      desc="  prefix pass 1/2"):
        for text in chunk[text_col].dropna().tolist():
            norm, _ = stage0_normalize(text)
            if isinstance(norm, str) and norm.strip():
                seed_counts[get_seed(norm)] += 1
    frequent = {s for s, c in seed_counts.items() if c >= PREFIX_MIN_COUNT}

    # Pass 2 — collect up to CAP normalized samples per frequent seed
    samples = defaultdict(list)
    for chunk in tqdm(pd.read_csv(filepath, usecols=[text_col],
                                  chunksize=PREFIX_CHUNKSIZE, on_bad_lines="skip"),
                      desc="  prefix pass 2/2"):
        for text in chunk[text_col].dropna().tolist():
            norm, _ = stage0_normalize(text)
            if not isinstance(norm, str) or not norm.strip():
                continue
            s = get_seed(norm)
            if s in frequent and len(samples[s]) < PREFIX_SAMPLES_CAP:
                samples[s].append(norm)

    index, seen, rejected = [], set(), 0
    for seed, texts in samples.items():
        lcp = longest_common_prefix_words(texts)
        wc = len(lcp.split())
        if wc < PREFIX_MIN_WORDS or wc > PREFIX_MAX_WORDS:
            if wc > PREFIX_MAX_WORDS:
                rejected += 1
            continue
        if lcp not in seen:
            seen.add(lcp)
            index.append((lcp, seed_counts[seed]))
    index.sort(key=lambda x: -len(x[0].split()))
    print(f"  Prefix index: {len(index)} kept, {rejected} rejected (>{PREFIX_MAX_WORDS}w).\n")
    return index


# ==========================================================
# GOLDEN TESTS
# ==========================================================
def run_golden_tests():
    print("Running golden tests...")
    fails = 0

    def check(name, cond, detail=""):
        nonlocal fails
        print(f"  {'PASS' if cond else 'FAIL'} {name}" + ("" if cond else f"  -> {detail}"))
        if not cond:
            fails += 1

    g, _ = stage0_normalize("the report saysEnlarge image apChina policy")
    check("deglue_case", "says enlarge" in g.lower(), g)

    d, _ = stage0_normalize("covid19 g20 summit boeing 737 crash")
    check("digits_removed", not any(c.isdigit() for c in d), d)

    o1, _ = stage0_normalize("new delhi : the minister spoke today about reforms")
    check("orphan_colon_cleaned", not o1.lower().startswith("new delhi :"), o1)

    check("prefix_guard_constant", PREFIX_MAX_WORDS == 25, "PREFIX_MAX_WORDS")

    ba, bah = stage2_inline("(ap) the state police advised drivers to be careful today")
    check("bare_agency", "inline:bare_agency" in bah, f"{ba} {bah}")

    so, soh = stage2_inline("the player rubbished the reports about a transfer --iansdm/pur/bg")
    check("agency_signoff", "inline:agency_signoff" in soh, f"{so} {soh}")

    lf, _ = stage2_inline("agberebi: nigerian forward sone aluko expressed his delight")
    check("leftover_frag", lf.lower().startswith("nigerian"), lf)

    bh, bhh = stage2_inline("the beautiful bride @brittneeshort cannot wait for the day")
    check("bare_handle", "inline:bare_handle" in bhh and "@brittneeshort" not in bh, bh)

    sp, sph = stage2_inline("join us pic. twitter. com/kytvx the campaign continues")
    check("spaced_pic_url", "inline:spaced_pic_url" in sph, f"{sp} {sph}")

    # de-hardcoded: any "<word> video:" label, not just "cmpd"
    gv, gvh = stage2_inline("peta says go vegan wxyz video: police chief responds to concerns")
    check("gallery_video_dehardcoded", "inline:gallery_video" in gvh, f"{gv} {gvh}")

    pc, pch = stage2_inline("china faces discrimination toggle caption mark schiefelbein/ap reports")
    check("photo_credit", "inline:photo_credit" in pch, f"{pc} {pch}")

    bs, _ = stage2_inline("By Monday, the council had finished its review of the plan.")
    check("byline_safe", bs.lower().startswith("by monday"), bs)

    # carousel: text from the seam token onward is truncated
    cz, czh = stage1_sliding(
        "what you need to know about the dam emergency pause "
        "video robbery suspect microbes in flux water gushes today", [])
    check("carousel_truncate",
          "sliding:carousel_truncate" in czh and "robbery" not in cz, cz)

    st, sth = stage3_regex_strip("citadel lowered its holdings in novanta (nasdaq:novt) this quarter")
    check("stock_ticker", "regex:stock_ticker" in sth and "nasdaq" not in st.lower(), st)

    oe, oeh = stage3_regex_strip("contact us at alex***@gmail.com for more information today")
    check("obfuscated_email", "regex:obfuscated_email" in oeh, f"{oe} {oeh}")

    # de-hardcoded: structural repeated-token schedule (not the word "players")
    rd = stage4_drop_verdict("episodes ( ) episodes ( ) episodes ( ) episodes ( ) here")
    check("release_schedule_dehardcoded", "drop:release_schedule" in rd, str(rd))
    # non-match guard: a normal sentence with a repeated word must NOT drop
    rd2 = stage4_drop_verdict("the team played well and the team will play again next week")
    check("release_schedule_safe", "drop:release_schedule" not in rd2, str(rd2))

    print(f"\nGolden tests: {17 - fails}/17 passed.\n")
    if fails:
        raise RuntimeError("Golden tests FAILED — fix rules before running.")


# ==========================================================
# FULL RUN — chunked, append-per-chunk, tqdm progress
# ==========================================================
def main():
    t_start = time.time()

    # ---- 1. golden tests ----
    run_golden_tests()

    # ---- 2. prefix index ----
    prefix_index = build_prefix_index(INPUT_CSV, TEXT_COLUMN)

    # ---- 3. chunked processing ----
    # fresh start: remove any output from a previous run
    for path in (DECISION_LOG, CLEANED_CSV):
        if os.path.exists(path):
            os.remove(path)

    total_rows = total_dropped = total_kept_cleaned = total_kept_orig = 0
    first_chunk = True

    print(f"Processing in chunks of {CHUNKSIZE:,} rows...")
    reader = pd.read_csv(INPUT_CSV, chunksize=CHUNKSIZE, on_bad_lines="skip")

    for chunk_no, chunk in enumerate(reader, 1):
        if TEXT_COLUMN not in chunk.columns:
            raise ValueError(f"Column '{TEXT_COLUMN}' not found. "
                             f"Available: {list(chunk.columns)}")

        decisions, cleaned_rows = [], []
        for idx, row in tqdm(chunk.iterrows(), total=len(chunk),
                             desc=f"chunk {chunk_no}", leave=False):
            r = process_row(row[TEXT_COLUMN], prefix_index)
            decisions.append({
                "row_index":   idx,
                "final_action": r["action"],
                "stage_tags":  r["tags"],
                "orig_len":    len(str(row[TEXT_COLUMN])),
                "cleaned_len": len(str(r["cleaned"])),
            })
            if r["action"] == "drop":
                total_dropped += 1
            else:
                if r["action"] == "keep_cleaned":
                    total_kept_cleaned += 1
                else:
                    total_kept_orig += 1
                out = row.to_dict()
                out[TEXT_COLUMN] = r["cleaned"]
                cleaned_rows.append(out)

        total_rows += len(chunk)

        # ---- append both files after EVERY chunk ----
        mode   = "w" if first_chunk else "a"
        header = first_chunk
        pd.DataFrame(decisions).to_csv(DECISION_LOG, mode=mode,
                                       header=header, index=False)
        if cleaned_rows:
            pd.DataFrame(cleaned_rows).to_csv(CLEANED_CSV, mode=mode,
                                              header=header, index=False)
        first_chunk = False

        drop_frac = total_dropped / total_rows
        elapsed = time.time() - t_start
        print(f"  chunk {chunk_no}: {total_rows:,} rows done | "
              f"dropped {total_dropped:,} ({drop_frac:.1%}) | "
              f"{elapsed:.0f}s elapsed")

        # ---- safety gate ----
        if drop_frac > MAX_DROP_FRACTION:
            raise RuntimeError(
                f"ABORTING: cumulative drop fraction {drop_frac:.1%} exceeds "
                f"{MAX_DROP_FRACTION:.0%}. A rule may be too broad. "
                f"Chunks written so far remain on disk; inspect {DECISION_LOG}.")

    # ---- summary ----
    print("\n" + "=" * 60)
    print("FULL RUN COMPLETE")
    print("=" * 60)
    print(f"  Total rows processed : {total_rows:,}")
    print(f"  Kept (original)      : {total_kept_orig:,}")
    print(f"  Kept (cleaned)       : {total_kept_cleaned:,}")
    print(f"  Dropped              : {total_dropped:,} "
          f"({total_dropped / total_rows:.1%})")
    print(f"  Elapsed              : {time.time() - t_start:.0f}s")
    print(f"\n  Decision log : {DECISION_LOG}")
    print(f"  Cleaned data : {CLEANED_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/CC_News_Project/Boilerplate_cleaning/cc_news_boilerplt_cleaned12.csv", nrows=100)
df

result: Track A is a five-stage cleaning pipeline for the text_clean column, run row-by-row in a fixed order: Stage 0 normalize fixes character-level damage from scraping (glued words like saysEnlarge, all digits removed for concept modeling, stranded punctuation cleaned); Stage 1 sliding truncates structural repetition that scraping introduced (duplicated sentences, sliding-headline carousels stitched together by a pause seam, cross-document boilerplate openings); Stage 2 inline strip removes embedded page furniture (wire datelines, bylines, share buttons, image captions, video-player text, copyright lines, bare twitter handles, social-media embed signatures); Stage 3 regex strip removes pattern-based noise (URLs, paywall blurbs, metadata tags, stock tickers, obfuscated emails, gibberish character runs); Stage 4 drop verdict is the only place a row gets removed, judging the already-cleaned text — empty after cleaning, mostly code, mostly digits/symbols, or a repetitive schedule list. Underpinning all of it: every action is tagged stage:reason, the design guarantees clean-first and drop-last (a row only dies if it's still junk after every rescue attempt), and the most aggressive structural rule — the pause-based carousel detector — was calibrated from your actual data rather than guessed.

On 596,272 rows the pipeline kept 82.8% as cleaned, 10.9% untouched, and dropped 6.3% — a healthy distribution that suggests rules are firing on real noise without being over-aggressive. Stage 0 normalization touched 81% of rows (digits removed alone on 474k rows), confirming the corpus had pervasive scraping artifacts. Sliding fired on 14.9% (mainly exact_repeat 54k times and prefix 28k times — the cross-doc boilerplate index is doing real work). Inline strip touched 12.5%, dominated by wire datelines (20k), bare twitter handles (15k), and image tags (14k). Regex strip touched only 3.3% — most pattern-noise had already been cleaned by earlier stages. Of the 37,503 drops, the overwhelming majority — 28,922 (77%) — were empty_after_strip, meaning those rows were boilerplate-and-noise all the way through and Stages 0–3 cleaned them to nothing; the rest split between code-dump rows (6,464), symbol-dump rows (2,151), and schedule lists (1,449). On the kept-cleaned set, the median row lost only 1.5% of its length and 90% of rows lost less than 27% — so for most articles the pipeline trimmed lightly, with the heavier shrinkage concentrated in the 10% that genuinely needed it. The carousel detector fired 4,689 times, which matches the order of magnitude you'd expect from the original CSV analysis (668 pause-bearing rows in the 3,549-row boilerplate-phrase file, scaling to the full corpus). One number worth flagging for next week: regex:url fired only once, meaning Stage 0's digit removal effectively destroyed URLs before the URL rule could see them — the URL rule has become dead code and should either be reordered before Stage 0 or retired. Otherwise the run looks clean and the cleaned output is ready for the next round (language filtering, Track B tag clouds, Phase B dedup).

for this week, we stop at cleaning the whole dataset remvoing further boiler plate things, and many other things, we will analyze what we remvoed and how much, i need to look into certain things before i can further move into cleaning, then we clean title_clean for the second last part. check for symbols etc, and numbers months, nouns and stuff from concept modeling perspective. we will also look into tag cloud, language, word limit. but before that we need to check some things we take this next week

week of June, 2026

1. https://siddharth1.medium.com/text-cleaning-in-practice-real-world-examples-and-best-practices-b4f52a45c6f8

2. https://medium.com/@shekhar.manna83/ml-ai-improving-text-classification-performance-with-crawled-data-ba55ec995aa7
3.